# 05. Touren: Rundreisen und Wege über alle Straßen

In [Notebook 04](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf2/04_lokale_suche.ipynb) hast du gelernt, eine Lösung durch kleine Änderungen zu verbessern. Hier wendest du das auf eine der berühmtesten Aufgaben der Informatik an: **eine möglichst kurze Rundreise** durch viele Orte. Dazu kommt ihre Schwester, bei der man nicht alle **Orte**, sondern alle **Straßen** abfahren muss. Beide Aufgaben stecken in vielen Aufgaben der 2. Runde, oft gut versteckt (ein Roboter, der Bäume gießt, ein Müllauto, ein Streufahrzeug, ein Flugzeug).

In diesem Notebook lernst du:

- eine **Rundreise** als Liste zu behandeln, ihre Länge zu berechnen und sie mit einem **Schiedsrichter** zu prüfen ([Notebook 01](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf2/01_pruefen_und_schranken.ipynb)),
- alle Rundreisen **durchzuprobieren** und zu sehen, wo das aufhört (n! wächst furchtbar schnell),
- zwei **Aufbauverfahren**: nächster Nachbar und billigste Einfügung,
- **2-opt** (ein Stück der Tour umdrehen) mit der Längenänderung in **O(1)** statt die ganze Tour neu zu rechnen, und **Or-opt** (ein Stück verschieben),
- die **exakte** Lösung kleiner Fälle mit **Bitmasken** (Verfahren von Held und Karp) und was sie über die Grenze zwischen „kleinen“ und „großen“ Fällen zeigt,
- die **Güte** zu messen: mit einer **unteren Schranke**, mit dem exakten Optimum und mit einer **Tabelle über viele Startwerte**,
- eine Tour über **alle Straßen** zu finden: den **Eulerkreis** (Verfahren von Hierholzer), und was man tut, wenn er nicht existiert (Straßen **doppelt** fahren, Kreuzungen mit **ungeradem Grad** paaren, kürzeste Wege mit **Dijkstra**),
- die Rundreise mit **MiniZinc** (`circuit`) als Gegenprobe zu lösen.

Das neutrale Beispiel für die Rundreise ist ein **Schulbus**, der morgens alle Haltestellen anfährt. Für die Straßen ist es ein **Briefträger**. Die Ecken wenden alles an: auf die Routen der **Gießroboter** (44. BwInf), auf die **Müllabfuhr** (40.) und, für Schnelle, auf **Weniger krumme Touren** (41.). Wenn du eine Aufgabe nicht schaffst, kopierst du die Lösung aus dem aufklappbaren Teil und führst sie aus, damit die späteren Zellen laufen.

## 1. Die Rundreise: Länge, Schiedsrichter, Durchprobieren

Ein Schulbus startet an Haltestelle 0, soll **jede** der `n` Haltestellen **genau einmal** anfahren und am Ende zur Haltestelle 0 zurückkehren. Die Haltestellen sind Punkte `(x, y)`, gefahren wird in der Luftlinie. Gesucht ist die **kürzeste** solche **Rundreise** (englisch *Traveling Salesman Problem*, **TSP**, „Handlungsreisender“).

Wie in [Notebook 01](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf2/01_pruefen_und_schranken.ipynb) bauen wir zuerst die Messgeräte. Eine **Tour** ist die Liste der Orte in der Reihenfolge, in der man sie besucht (`[0, 3, 1, 2]`: von 0 nach 3, nach 1, nach 2 und zurück nach 0). Die Abstände rechnen wir **einmal** in einer Tabelle `d` aus, `d[i][j]` ist der Abstand von Ort `i` zu Ort `j`. Dann kostet jede Abfrage nur einen Tabellenzugriff, und alle Verfahren hier brauchen nur `d`, nicht die Koordinaten. `zufalls_orte(startwert, n)` erzeugt immer dieselben Orte zu einem Startwert (damit alle dieselben Zahlen sehen), `ganze_matrix` brauchen wir erst bei MiniZinc.

In [ ]:
import math
import random
import itertools


def zufalls_orte(startwert, n):
    """n Orte mit ganzzahligen Koordinaten von 0 bis 100, bei festem Startwert immer dieselben."""
    zufall = random.Random(startwert)
    orte = []
    for i in range(n):
        orte.append((zufall.randint(0, 100), zufall.randint(0, 100)))
    return orte


def abstandsmatrix(orte):
    """d[i][j] ist der Abstand (Luftlinie) von Ort i zu Ort j."""
    d = []
    for a in orte:
        zeile = []
        for b in orte:
            zeile.append(math.dist(a, b))
        d.append(zeile)
    return d


def ganze_matrix(orte):
    """Wie abstandsmatrix, aber auf Zehntel gerundete ganze Zahlen (für MiniZinc, das mit ganzen Zahlen rechnet)."""
    d = abstandsmatrix(orte)
    for zeile in d:
        for j in range(len(zeile)):
            zeile[j] = round(10 * zeile[j])
    return d

#### Aufgabe 1.1

Schreibe `tour_laenge(d, tour)`: die Länge der Rundreise, also die Summe der Abstände von jedem Ort zum nächsten **und vom letzten zurück zum ersten**. Teste sie am Quadrat mit der Seitenlänge 10: Die Tour `[0, 1, 2, 3]` geht um das Quadrat herum, `[0, 2, 1, 3]` fährt zweimal quer über die Diagonale.

**Erwartete Ausgabe:**
```
40.0
48.28
```

In [ ]:
def tour_laenge(d, tour):
    # TODO
    return 0


quadrat = [(0, 0), (10, 0), (10, 10), (0, 10)]
d4 = abstandsmatrix(quadrat)
print(round(tour_laenge(d4, [0, 1, 2, 3]), 2))
print(round(tour_laenge(d4, [0, 2, 1, 3]), 2))

<details>
<summary>Tipp anzeigen</summary>

Zähle `k` von 0 bis `n - 1` und addiere `d[tour[k]][tour[k + 1]]`. Für den letzten Ort ist der „nächste“ der erste: Mit `(k + 1) % n` springt die Zahl wieder auf 0.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def tour_laenge(d, tour):
    """Länge der Rundreise: von jedem Ort zum nächsten, vom letzten zurück zum ersten."""
    summe = 0
    for k in range(len(tour)):
        summe = summe + d[tour[k]][tour[(k + 1) % len(tour)]]
    return summe


quadrat = [(0, 0), (10, 0), (10, 10), (0, 10)]
d4 = abstandsmatrix(quadrat)
print(round(tour_laenge(d4, [0, 1, 2, 3]), 2))
print(round(tour_laenge(d4, [0, 2, 1, 3]), 2))
```

Der Rest `% len(tour)` schließt den Kreis. Die Diagonalen sind länger als die Seiten, deshalb ist die Tour um das Quadrat (40) kürzer als die mit den Diagonalen (etwa 48,28).

</details>

#### Aufgabe 1.2

Schreibe den **Schiedsrichter** `pruefe_tour(d, tour)`: Er gibt `'ok'` zurück, wenn die Tour **jeden** Ort **genau einmal** enthält, sonst eine Beschreibung des ersten Fehlers (falsche Länge, ein Ort kommt doppelt vor, ein Ort gibt es nicht). Jedes Verfahren in diesem Notebook wird damit geprüft.

**Erwartete Ausgabe:**
```
[0, 1, 2, 3] ok
[0, 1, 2] die Tour hat 3 Orte, es sollten 4 sein
[0, 1, 1, 3] Ort 1 kommt mehr als einmal vor
[0, 1, 2, 4] Ort 4 gibt es nicht
```

In [ ]:
def pruefe_tour(d, tour):
    # TODO
    return 'ok'


for tour in ([0, 1, 2, 3], [0, 1, 2], [0, 1, 1, 3], [0, 1, 2, 4]):
    print(tour, pruefe_tour(d4, tour))

<details>
<summary>Tipp anzeigen</summary>

Erst die Länge prüfen. Dann mit einer Menge (`set`) merken, welche Orte schon vorkamen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def pruefe_tour(d, tour):
    """Schiedsrichter: 'ok' oder die Beschreibung des ersten Fehlers."""
    n = len(d)
    if len(tour) != n:
        return f'die Tour hat {len(tour)} Orte, es sollten {n} sein'
    gesehen = set()
    for ort in tour:
        if ort < 0 or ort >= n:
            return f'Ort {ort} gibt es nicht'
        if ort in gesehen:
            return f'Ort {ort} kommt mehr als einmal vor'
        gesehen.add(ort)
    return 'ok'


for tour in ([0, 1, 2, 3], [0, 1, 2], [0, 1, 1, 3], [0, 1, 2, 4]):
    print(tour, pruefe_tour(d4, tour))
```

</details>

Die einfachste Idee: **alle Rundreisen durchprobieren** (Brute Force wie in [Notebook 06](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf2/06_backtracking_gross.ipynb)) und die kürzeste nehmen. Dazu zwei Beobachtungen, die den Aufwand halbieren und durch `n` teilen:

- Eine Rundreise hat **keinen Anfang**. `[0, 1, 2, 3]`, `[1, 2, 3, 0]` und `[2, 3, 0, 1]` sind dieselbe Rundreise. Wir legen Ort 0 an den Anfang fest und ordnen nur die **übrigen** `n − 1` Orte an.
- Eine Rundreise kann man **rückwärts** fahren: `[0, 1, 2, 3]` und `[0, 3, 2, 1]` sind gleich lang. Wir nehmen von jedem solchen Paar nur eine.

Zusammen sind das `(n − 1)! / 2` Rundreisen. Zum Aufzählen aller Anordnungen gibt es `itertools.permutations`.

#### Frage 1.3

Wie viele verschiedene Rundreisen gibt es bei `n = 10`, bei `n = 15` und bei `n = 20` Orten? Wie lange würde das Durchprobieren bei `n = 20` dauern, wenn dein Rechner **eine Million** Rundreisen pro Sekunde prüft? Gib eine grobe Zeit in Jahren an.

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

`(n − 1)! / 2`:

- `n = 10`: 9! / 2 = **181 440**
- `n = 15`: 14! / 2 = **43 589 145 600** (rund 4,4 · 10¹⁰)
- `n = 20`: 19! / 2 = **60 822 550 204 416 000** (rund 6 · 10¹⁶)

Bei einer Million pro Sekunde sind das 6 · 10¹⁰ Sekunden. Ein Jahr hat etwa 3 · 10⁷ Sekunden, es dauert also rund **1 900 Jahre**. Das Durchprobieren geht also nur bis etwa `n = 10`, mit Geduld bis `n = 12`. Dass die Zahl so schnell wächst, liegt an der Fakultät: Jeder neue Ort multipliziert die Zahl der Möglichkeiten mit der Zahl der Orte.

</details>

#### Aufgabe 1.4

Schreibe `tour_brute_force(d)`: Ort 0 steht am Anfang, probiere alle Anordnungen der übrigen Orte mit `itertools.permutations`, überspringe die rückwärts gefahrenen (`rest[0] > rest[-1]`) und merke dir die kürzeste Tour. Gib `(kürzeste Länge, Tour, Zahl der geprüften Touren)` zurück. Zähle mit, wie viele Touren du geprüft hast, und vergleiche mit `(n − 1)! / 2`.

**Erwartete Ausgabe:**
```
5 279.8 [0, 2, 1, 4, 3] 12
6 285.2 [0, 2, 1, 5, 4, 3] 60
7 288.8 [0, 2, 1, 6, 5, 4, 3] 360
8 292.2 [0, 3, 4, 5, 6, 1, 2, 7] 2520
```

In [ ]:
def tour_brute_force(d):
    n = len(d)
    # TODO
    return 0, [], 0


for n in range(5, 9):
    d = abstandsmatrix(zufalls_orte(1, n))
    laenge, tour, geprueft = tour_brute_force(d)
    print(n, round(laenge, 1), tour, geprueft)

<details>
<summary>Tipp anzeigen</summary>

`for rest in itertools.permutations(range(1, n)):` liefert jede Anordnung als Tupel. Daraus wird `tour = [0] + list(rest)`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def tour_brute_force(d):
    """Probiert alle Rundreisen durch. Ort 0 steht fest am Anfang, jede Tour gilt nur einmal (nicht auch rückwärts).
    Gibt (kürzeste Länge, Tour, Zahl der geprüften Touren) zurück."""
    n = len(d)
    beste_laenge = None
    beste_tour = None
    geprueft = 0
    for rest in itertools.permutations(range(1, n)):
        if rest[0] > rest[-1]:                  # dieselbe Tour rückwärts: schon gesehen
            continue
        tour = [0] + list(rest)
        geprueft = geprueft + 1
        laenge = tour_laenge(d, tour)
        if beste_laenge is None or laenge < beste_laenge:
            beste_laenge = laenge
            beste_tour = tour
    return beste_laenge, beste_tour, geprueft


for n in range(5, 9):
    d = abstandsmatrix(zufalls_orte(1, n))
    laenge, tour, geprueft = tour_brute_force(d)
    print(n, round(laenge, 1), tour, geprueft)
```

Die Zahlen der geprüften Touren sind 12, 60, 360 und 2 520, also `(n − 1)! / 2` für `n = 5` bis `8`. Jeder weitere Ort multipliziert mit `n − 1`.

</details>

## 2. Zwei Aufbauverfahren

Für größere Fälle brauchen wir **Heuristiken** wie in [Notebook 03](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf2/03_heuristiken_bauen.ipynb). Zwei einfache Regeln, eine Tour aufzubauen:

- **Nächster Nachbar:** Starte an Ort 0 und fahre immer zum **nächsten noch nicht besuchten** Ort. Am Ende kehrst du zurück.
- **Billigste Einfügung:** Beginne mit einer winzigen Tour aus Ort 0 und seinem nächsten Nachbarn. Dann nimm unter **allen** noch freien Orten den, den man an **irgendeiner** Stelle der Tour mit dem **kleinsten Umweg** einfügen kann, und füge ihn dort ein. Der Umweg beim Einfügen von `c` zwischen `a` und `b` ist `d[a][c] + d[c][b] − d[a][b]`.

#### Aufgabe 2.1

Schreibe `nachster_nachbar(d, start=0)`. Teste es an den Orten mit den Startwerten 1 bis 5 (jeweils 8 Orte): Die Ausgabe zeigt die optimale Länge (Brute Force aus 1.4), die Länge der Tour des Verfahrens und das Urteil des Schiedsrichters.

**Erwartete Ausgabe:**
```
1 292.2 336.3 ok
2 277.9 296.2 ok
3 199.2 206.0 ok
4 228.1 253.1 ok
5 316.8 354.6 ok
```

In [ ]:
def nachster_nachbar(d, start=0):
    n = len(d)
    # TODO
    return list(range(n))


for startwert in range(1, 6):
    d = abstandsmatrix(zufalls_orte(startwert, 8))
    optimum = tour_brute_force(d)[0]
    tour = nachster_nachbar(d)
    print(startwert, round(optimum, 1), round(tour_laenge(d, tour), 1), pruefe_tour(d, tour))

<details>
<summary>Tipp anzeigen</summary>

Führe eine Liste `besucht` mit `True`/`False` je Ort. In einer Schleife suchst du unter den Orten `j` mit `besucht[j] == False` den mit kleinstem `d[letzter][j]`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def nachster_nachbar(d, start=0):
    """Von jedem Ort zum nächsten noch nicht besuchten Ort."""
    n = len(d)
    tour = [start]
    besucht = [False] * n
    besucht[start] = True
    while len(tour) < n:
        letzter = tour[-1]
        bester = None
        for j in range(n):
            if not besucht[j] and (bester is None or d[letzter][j] < d[letzter][bester]):
                bester = j
        tour.append(bester)
        besucht[bester] = True
    return tour


for startwert in range(1, 6):
    d = abstandsmatrix(zufalls_orte(startwert, 8))
    optimum = tour_brute_force(d)[0]
    tour = nachster_nachbar(d)
    print(startwert, round(optimum, 1), round(tour_laenge(d, tour), 1), pruefe_tour(d, tour))
```

</details>

#### Aufgabe 2.2

Schreibe `billigste_einfuegung(d)` nach der Beschreibung oben (Start mit Ort 0 und seinem nächsten Nachbarn; dann immer die billigste Einfügung unter allen freien Orten und allen Stellen). Mit `tour.insert(position, ort)` setzt du einen Ort an die Stelle `position` der Liste. Teste an denselben Orten wie 2.1 und gib zusätzlich die Länge der Tour des nächsten Nachbarn aus.

**Erwartete Ausgabe:**
```
1 292.2 336.3 324.4 ok
2 277.9 296.2 277.9 ok
3 199.2 206.0 199.2 ok
4 228.1 253.1 228.1 ok
5 316.8 354.6 348.9 ok
```

In [ ]:
def billigste_einfuegung(d):
    n = len(d)
    # TODO
    return list(range(n))


for startwert in range(1, 6):
    d = abstandsmatrix(zufalls_orte(startwert, 8))
    optimum = tour_brute_force(d)[0]
    nn = nachster_nachbar(d)
    ei = billigste_einfuegung(d)
    print(startwert, round(optimum, 1), round(tour_laenge(d, nn), 1), round(tour_laenge(d, ei), 1), pruefe_tour(d, ei))

<details>
<summary>Tipp anzeigen</summary>

Für jeden freien Ort `c` und jede Position `k` der Tour (zwischen `tour[k]` und `tour[(k + 1) % len(tour)]`) rechnest du den Umweg aus und merkst dir das Minimum samt `c` und `k + 1`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def nachster_nachbar(d, start=0):
    """Von jedem Ort zum nächsten noch nicht besuchten Ort."""
    n = len(d)
    tour = [start]
    besucht = [False] * n
    besucht[start] = True
    while len(tour) < n:
        letzter = tour[-1]
        bester = None
        for j in range(n):
            if not besucht[j] and (bester is None or d[letzter][j] < d[letzter][bester]):
                bester = j
        tour.append(bester)
        besucht[bester] = True
    return tour


def billigste_einfuegung(d):
    """Beginnt mit Ort 0 und seinem nächsten Nachbarn und fügt immer den Ort an der Stelle ein, wo der Umweg am kleinsten ist."""
    n = len(d)
    naechster = 1
    for j in range(1, n):
        if d[0][j] < d[0][naechster]:
            naechster = j
    tour = [0, naechster]
    frei = []
    for j in range(n):
        if j != 0 and j != naechster:
            frei.append(j)
    while len(frei) > 0:
        bester_umweg = None
        for c in frei:
            for k in range(len(tour)):
                a = tour[k]
                b = tour[(k + 1) % len(tour)]
                umweg = d[a][c] + d[c][b] - d[a][b]
                if bester_umweg is None or umweg < bester_umweg:
                    bester_umweg = umweg
                    bester_ort = c
                    beste_stelle = k + 1          # c kommt hinter Position k
        tour.insert(beste_stelle, bester_ort)
        frei.remove(bester_ort)
    return tour



for startwert in range(1, 6):
    d = abstandsmatrix(zufalls_orte(startwert, 8))
    optimum = tour_brute_force(d)[0]
    nn = nachster_nachbar(d)
    ei = billigste_einfuegung(d)
    print(startwert, round(optimum, 1), round(tour_laenge(d, nn), 1), round(tour_laenge(d, ei), 1), pruefe_tour(d, ei))
```

</details>

#### Frage 2.3

1. Wie viele Schritte braucht `nachster_nachbar` für `n` Orte, und wie viele `billigste_einfuegung` (in der Schreibweise mit O)? Begründe mit den Schleifen.
2. Wie oft finden die beiden Verfahren in der Ausgabe von 2.2 das Optimum? Was heißt es, dass die Länge nie **unter** dem Optimum liegt?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. **Nächster Nachbar:** `n − 1` Schritte, in jedem suchst du unter bis zu `n` Orten: **O(n²)**. **Billigste Einfügung** (so geschrieben): `n − 2` Schritte, in jedem probierst du jeden freien Ort (bis `n`) an jeder Stelle der Tour (bis `n`): **O(n³)**. Das ist für `n = 100` noch eine Million Umwege und für `n = 1000` schon eine Milliarde.
2. Der nächste Nachbar trifft das Optimum bei 0 von 5, die billigste Einfügung bei 3 von 5 Fällen. Dass kein Verfahren unter dem Optimum liegt, ist keine Leistung, sondern ein **Test des Schiedsrichters**: Das Optimum ist die kürzeste mögliche Länge. Läge ein Verfahren darunter, wäre die Tour ungültig oder die Brute-Force-Suche falsch.

</details>

## 3. Verbessern mit 2-opt und Or-opt

Beide Aufbauverfahren machen Fehler, die man **sieht**: Die Tour kreuzt sich selbst. Eine Tour mit einer Kreuzung ist **nie** kürzer als eine ohne (Dreiecksungleichung: Zwei Seiten eines Dreiecks sind zusammen länger als die dritte). Das Umdrehen eines Stücks beseitigt die Kreuzung.

**2-opt:** Zwei Kanten der Tour, `(a, b)` und `(c, e)`, werden entfernt und durch `(a, c)` und `(b, e)` ersetzt. Das entspricht dem **Umkehren** des Stücks von `b` bis `c`:

```
vorher:   ... a b ... c e ...      (Kanten a-b und c-e)
nachher:  ... a c ... b e ...      (Kanten a-c und b-e; das Stück b ... c ist umgedreht)
```

Das ist die **Nachbarschaft** aus [Notebook 04](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf2/04_lokale_suche.ipynb): Ein Zug ist ein Paar `(i, j)` von Positionen. Und es gilt wieder der Trick der **inkrementellen Bewertung**: Von der ganzen Tour ändern sich nur **zwei Kanten**, die Längenänderung ist `d[a][c] + d[b][e] − d[a][b] − d[c][e]`. Sie kostet **O(1)** statt O(n) für die ganze Tour neu rechnen. Ist sie negativ, wird die Tour kürzer.

#### Aufgabe 3.1

Schreibe `delta_zwei_opt(d, tour, i, j)`: die Längenänderung, wenn man das Stück von Position `i + 1` bis `j` umkehrt. Dabei sind `a = tour[i]`, `b = tour[i + 1]`, `c = tour[j]` und `e = tour[(j + 1) % n]`. Prüfe sie am Quadrat mit der Kreuzung `[0, 2, 1, 3]`: Die vorhergesagte und die wirkliche Änderung müssen übereinstimmen.

**Erwartete Ausgabe:**
```
[0, 1, 2, 3]
-8.28
-8.28
```

In [ ]:
ZAEHLER = {'delta': 0, 'runden': 0}


def delta_zwei_opt(d, tour, i, j):
    ZAEHLER['delta'] = ZAEHLER['delta'] + 1
    # TODO
    return 0


tour = [0, 2, 1, 3]
neu = tour[:1] + tour[1:3][::-1] + tour[3:]            # Stück von Position 1 bis 2 umkehren (i = 0, j = 2)
print(neu)
print(round(delta_zwei_opt(d4, tour, 0, 2), 2))
print(round(tour_laenge(d4, neu) - tour_laenge(d4, tour), 2))

<details>
<summary>Tipp anzeigen</summary>

Die alten Kanten sind `d[a][b]` und `d[c][e]`, die neuen `d[a][c]` und `d[b][e]`. Die Änderung ist neu minus alt.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
ZAEHLER = {'delta': 0, 'runden': 0}      # wie oft wir eine Änderung bewertet haben, wie viele Durchläufe es gab


def delta_zwei_opt(d, tour, i, j):
    """Wie viel länger wird die Tour, wenn man das Stück von Position i+1 bis j umkehrt? (negativ = kürzer)
    Es ändern sich nur zwei Kanten: (a, b) und (c, e) werden zu (a, c) und (b, e)."""
    ZAEHLER['delta'] = ZAEHLER['delta'] + 1
    n = len(tour)
    a = tour[i]
    b = tour[i + 1]
    c = tour[j]
    e = tour[(j + 1) % n]
    return d[a][c] + d[b][e] - d[a][b] - d[c][e]


tour = [0, 2, 1, 3]
neu = tour[:1] + tour[1:3][::-1] + tour[3:]            # Stück von Position 1 bis 2 umkehren (i = 0, j = 2)
print(neu)
print(round(delta_zwei_opt(d4, tour, 0, 2), 2))
print(round(tour_laenge(d4, neu) - tour_laenge(d4, tour), 2))
```

Das Kreuz verschwindet: Die beiden Diagonalen (je etwa 14,14) werden durch zwei Seiten (je 10) ersetzt. Das spart etwa 8,28, die Tour wird um so viel **kürzer** (Änderung −8,28).

</details>

#### Aufgabe 3.2

Schreibe `zwei_opt(d, tour)`: Probiere alle Paare `i < j` (mit `j ≥ i + 2`, und nicht `i == 0` zusammen mit `j == n − 1`, das wäre die ganze Tour rückwärts). Ist `delta_zwei_opt(...) < −0,000000001` (eine winzige Toleranz wegen der Kommazahlen), kehrst du das Stück um (`tour[i + 1:j + 1] = tour[i + 1:j + 1][::-1]`) und machst weiter. Wiederhole, bis in einem ganzen Durchlauf nichts mehr besser wurde. `ZAEHLER['runden']` zählt die Durchläufe.

Teste an 30 Orten mit den Startwerten 1 bis 4: Länge nach dem nächsten Nachbarn, Länge danach, Urteil des Schiedsrichters, und am Ende, wie oft du eine Änderung bewertet hast.

**Erwartete Ausgabe:**
```
1 652.7 557.2 ok
2 586.2 495.8 ok
3 546.4 509.9 ok
4 563.3 501.7 ok
{'delta': 4455, 'runden': 11}
```

In [ ]:
def zwei_opt(d, tour):
    tour = tour[:]
    n = len(tour)
    # TODO
    return tour


for startwert in range(1, 5):
    d = abstandsmatrix(zufalls_orte(startwert, 30))
    start = nachster_nachbar(d)
    besser = zwei_opt(d, start)
    print(startwert, round(tour_laenge(d, start), 1), round(tour_laenge(d, besser), 1), pruefe_tour(d, besser))
print(ZAEHLER)

<details>
<summary>Tipp anzeigen</summary>

Eine `while`-Schleife mit einer Variablen `verbessert`, die du vor jedem Durchlauf auf `False` setzt und bei jeder Änderung auf `True`. Zwei verschachtelte `for`-Schleifen für `i` und `j`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
ZAEHLER = {'delta': 0, 'runden': 0}      # wie oft wir eine Änderung bewertet haben, wie viele Durchläufe es gab


def delta_zwei_opt(d, tour, i, j):
    """Wie viel länger wird die Tour, wenn man das Stück von Position i+1 bis j umkehrt? (negativ = kürzer)
    Es ändern sich nur zwei Kanten: (a, b) und (c, e) werden zu (a, c) und (b, e)."""
    ZAEHLER['delta'] = ZAEHLER['delta'] + 1
    n = len(tour)
    a = tour[i]
    b = tour[i + 1]
    c = tour[j]
    e = tour[(j + 1) % n]
    return d[a][c] + d[b][e] - d[a][b] - d[c][e]


def zwei_opt(d, tour):
    """Verbessert die Tour, solange es ein Umkehren gibt, das sie kürzer macht (erste Verbesserung)."""
    tour = tour[:]
    n = len(tour)
    verbessert = True
    while verbessert:
        verbessert = False
        ZAEHLER['runden'] = ZAEHLER['runden'] + 1
        for i in range(n - 1):
            for j in range(i + 2, n):
                if i == 0 and j == n - 1:        # dann wäre es die ganze Tour rückwärts
                    continue
                if delta_zwei_opt(d, tour, i, j) < -0.000000001:
                    tour[i + 1:j + 1] = tour[i + 1:j + 1][::-1]
                    verbessert = True
    return tour



for startwert in range(1, 5):
    d = abstandsmatrix(zufalls_orte(startwert, 30))
    start = nachster_nachbar(d)
    besser = zwei_opt(d, start)
    print(startwert, round(tour_laenge(d, start), 1), round(tour_laenge(d, besser), 1), pruefe_tour(d, besser))
print(ZAEHLER)
```

Der Schiedsrichter sagt bei allen vier `ok`: Das Umkehren eines Stücks erzeugt wieder eine gültige Tour. Die Zahl der bewerteten Änderungen ist die Zahl der Durchläufe mal 405 Paare (bei `n = 30` Orten gibt es 435 Paare `i < j`, ohne die 29 Nachbarn und die eine ganze Tour bleiben 405): 11 Durchläufe, 4 455 Bewertungen. Mit voller Neuberechnung der Länge hätte jede Bewertung 30 Abstände addiert.

</details>

**Finde den Fehler.** Ein häufiger Fehler beim 2-opt steckt nicht in der Formel, sondern im **Umkehren**. Die Zelle kehrt das Stück um und vergleicht die vorhergesagte mit der wirklichen Änderung (wie der Schiedsrichter in Notebook 04). Führe sie aus.

In [ ]:
tour = [0, 2, 1, 3]
i, j = 0, 2                                              # kehre das Stück von Position i + 1 bis j um
vorhergesagt = delta_zwei_opt(d4, tour, i, j)
falsch = tour[:i + 1] + tour[i + 1:j][::-1] + tour[j:]
print(falsch)
print('vorhergesagt', round(vorhergesagt, 2), ' wirklich', round(tour_laenge(d4, falsch) - tour_laenge(d4, tour), 2))

[0, 2, 1, 3]
vorhergesagt -8.28  wirklich 0.0


#### Frage 3.3

Was ist falsch? Warum bemerkt der Schiedsrichter `pruefe_tour` den Fehler **nicht**? Was würde in `zwei_opt` passieren, wenn man das Umkehren so schriebe?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Tipp anzeigen</summary>

Die Ausgabe zeigt: vorhergesagt −8,28, wirklich 0. Vergleiche die Tour davor und danach.

</details>

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

Die Scheibe `tour[i + 1:j]` hört **vor** Position `j` auf (das Ende einer Scheibe ist nie dabei). Das Stück, das umgekehrt werden soll, reicht aber **bis einschließlich** `j`: richtig ist `tour[i + 1:j + 1]`. Hier besteht `tour[1:2]` nur aus einem Ort, das Umkehren ändert nichts.

Der Schiedsrichter `pruefe_tour` prüft nur, ob jeder Ort **genau einmal** vorkommt. Das tut die unveränderte Tour. Er prüft nicht, ob die Länge so ist, wie das Delta behauptet. Dafür braucht man die Gegenprobe wie in der Zelle: **Delta gegen die neu berechnete Länge** an ein paar Zügen vergleichen.

In `zwei_opt` würde die Schleife den Zug als Verbesserung ansehen (das Delta ist negativ), die Tour bliebe aber gleich, und beim nächsten Durchlauf fände sie denselben Zug wieder: eine **Endlosschleife**.

</details>

**Or-opt.** 2-opt dreht Stücke um, verschiebt aber keinen einzelnen Ort an eine ganz andere Stelle. Das erledigt **Or-opt**: Ein Stück aus **1, 2 oder 3 Orten** wird herausgenommen und (vorwärts oder rückwärts) an der günstigsten anderen Stelle wieder eingesetzt. Die Ersparnis beim Herausnehmen ist `d[davor][erster] + d[letzter][danach] − d[davor][danach]`, die Kosten beim Einsetzen zwischen `a` und `b` sind `d[a][erster] + d[letzter][b] − d[a][b]` (oder umgekehrt, wenn man das Stück rückwärts einsetzt). Auch das sind nur ein paar Tabellenzugriffe, also **O(1)** je Zug. Wir lesen die fertige Funktion nur, sie besteht aus denselben Bausteinen. `zwei_opt_und_or_opt` wechselt beide ab, bis keines mehr etwas findet. Beide Nachbarschaften zusammen sind größer, die Tour bleibt also **seltener in einem lokalen Optimum** hängen ([Notebook 04](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf2/04_lokale_suche.ipynb)).

In [ ]:
def or_opt(d, tour):
    """Nimmt ein Stück aus 1, 2 oder 3 Orten heraus und setzt es (vorwärts oder rückwärts) an einer anderen Stelle wieder ein,
    solange das die Tour kürzer macht. Ort 0 bleibt dabei an Position 0."""
    tour = tour[:]
    n = len(tour)
    verbessert = True
    while verbessert:
        verbessert = False
        for laenge in (1, 2, 3):
            for i in range(1, n - laenge + 1):
                stueck = tour[i:i + laenge]
                davor = tour[i - 1]
                danach = tour[(i + laenge) % n]
                gewinn = d[davor][stueck[0]] + d[stueck[-1]][danach] - d[davor][danach]     # so viel sparen wir beim Herausnehmen
                rest = tour[:i] + tour[i + laenge:]
                for k in range(len(rest)):
                    a = rest[k]
                    b = rest[(k + 1) % len(rest)]
                    kosten_vor = d[a][stueck[0]] + d[stueck[-1]][b] - d[a][b]
                    kosten_rueck = d[a][stueck[-1]] + d[stueck[0]][b] - d[a][b]
                    if min(kosten_vor, kosten_rueck) < gewinn - 0.000000001:
                        if kosten_vor <= kosten_rueck:
                            tour = rest[:k + 1] + stueck + rest[k + 1:]
                        else:
                            tour = rest[:k + 1] + stueck[::-1] + rest[k + 1:]
                        verbessert = True
                        break
                if verbessert:
                    break
            if verbessert:
                break
    return tour


def zwei_opt_und_or_opt(d, tour):
    """Abwechselnd 2-opt und Or-opt, bis keines mehr etwas findet."""
    while True:
        neu = or_opt(d, zwei_opt(d, tour))
        if tour_laenge(d, neu) > tour_laenge(d, tour) - 0.000000001:
            return neu
        tour = neu

## 4. Die exakte Lösung kleiner Fälle: Held-Karp mit Bitmasken

Durchprobieren geht bis etwa `n = 10`. Es geht aber **viel besser**, wenn man **Teilergebnisse merkt** ([Notebook 10](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/10_merken.ipynb) und [Notebook 11](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/11_tabellen_dp.ipynb) von learn_bwinf1). Die Beobachtung von **Held und Karp**:

> Hat man schon alle Orte einer Menge `S` besucht und steht jetzt im Ort `j` von `S`, dann hängt die **Restfahrt** nur von `S` und `j` ab, nicht davon, **in welcher Reihenfolge** man die Orte von `S` besucht hat.

Also genügt es, für jede Menge `S` (die Ort 0 enthält) und jeden Ort `j` in `S` den **kürzesten Weg** zu merken, der bei Ort 0 beginnt, genau die Orte von `S` besucht und in `j` endet:

`bester[S][j] = kleinstes über alle k in S ohne j von  bester[S ohne j][k] + d[k][j]`.

Am Ende nimmt man das Beste von `bester[alle Orte][j] + d[j][0]` über alle `j`. Das sind `2ⁿ` Mengen mal `n` Endorte mal `n` Vorgänger: **O(2ⁿ · n²)** statt `n!`.

**Bitmasken.** Wie speichert man eine **Menge von Orten** als Schlüssel in einer Tabelle? Als **ganze Zahl**: Das Bit Nummer `i` der Zahl ist 1, wenn Ort `i` in der Menge ist. Die Menge `{0, 3}` ist `1 + 8 = 9`, in Binärschreibweise `1001`. Alle Mengen von `n` Orten sind die Zahlen von `0` bis `2ⁿ − 1`, man kann sie also als **Index einer Liste** benutzen. Vier Grundoperationen:

| Gedanke | Python | Beispiel |
|---|---|---|
| die Menge `{i}` | `1 << i` (schiebt die 1 um `i` Stellen nach links) | `1 << 3` ist 8 |
| `i` hinzufügen | `maske \| (1 << i)` | |
| ist `i` drin? | `maske & (1 << i) != 0` | |
| `i` entfernen (wenn drin) | `maske ^ (1 << i)` | |

Dabei sind `|` (oder), `&` (und) und `^` (entweder-oder) Operationen **auf den einzelnen Bits** der Zahlen. Wichtig ist, dass jede neue Menge durch Hinzufügen eines Bits **größer** wird. Wenn man die Zahlen von klein nach groß durchgeht, sind alle Teilmengen vor ihren Obermengen dran.

#### Aufgabe 4.1: Was wird ausgegeben?

Überlege zuerst, was der Code ausgibt, und schreibe es auf. Führe die Zelle erst danach aus.

In [ ]:
maske = 0
maske = maske | (1 << 0)
maske = maske | (1 << 3)
print(maske, bin(maske))
print(maske & (1 << 3) != 0, maske & (1 << 2) != 0)
maske = maske ^ (1 << 0)
print(maske)
for i in range(4):
    print(i, 1 << i)

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

**Ausgabe:**
```
9 0b1001
True False
8
0 1
1 2
2 4
3 8
```

`1 << 0` ist 1, `1 << 3` ist 8, die Maske ist `1 | 8 = 9` (binär `1001`, die Bits 0 und 3 sind gesetzt). Ort 3 ist drin (`True`), Ort 2 nicht (`False`). `9 ^ 1` nimmt das Bit 0 weg und ergibt 8. Die letzte Schleife zeigt die Mengen `{0}`, `{1}`, `{2}`, `{3}` als Zahlen 1, 2, 4, 8: **Zweierpotenzen**.

</details>

#### Aufgabe 4.2

Schreibe vier kleine Funktionen für Mengen als Bitmaske: `enthaelt(maske, i)` (ist Ort `i` drin?), `mit(maske, i)` (die Menge mit `i`), `ohne(maske, i)` (die Menge ohne `i`) und `als_liste(maske, n)` (die Orte der Menge als Liste, von klein nach groß, bei höchstens `n` Orten).

**Erwartete Ausgabe:**
```
82 [1, 4, 6]
True False
66 [1, 6]
```

In [ ]:
def enthaelt(maske, i):
    # TODO
    return False


def mit(maske, i):
    # TODO
    return maske


def ohne(maske, i):
    # TODO
    return maske


def als_liste(maske, n):
    # TODO
    return []


m = mit(mit(mit(0, 1), 4), 6)
print(m, als_liste(m, 8))
print(enthaelt(m, 4), enthaelt(m, 5))
m = ohne(m, 4)
print(m, als_liste(m, 8))

<details>
<summary>Tipp anzeigen</summary>

`enthaelt` ist `maske & (1 << i) != 0`. Für `ohne` hilft `~x` (alle Bits von `x` umgedreht): `maske & ~(1 << i)` behält alle Bits außer Bit `i`. (Mit `^` geht es nur, wenn `i` sicher drin ist.)

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def enthaelt(maske, i):
    return maske & (1 << i) != 0


def mit(maske, i):
    return maske | (1 << i)


def ohne(maske, i):
    return maske & ~(1 << i)               # alle Bits behalten außer Bit i


def als_liste(maske, n):
    liste = []
    for i in range(n):
        if enthaelt(maske, i):
            liste.append(i)
    return liste


m = mit(mit(mit(0, 1), 4), 6)
print(m, als_liste(m, 8))
print(enthaelt(m, 4), enthaelt(m, 5))
m = ohne(m, 4)
print(m, als_liste(m, 8))
```

`2 + 16 + 64 = 82`. Ohne Ort 4 bleibt `2 + 64 = 66`.

</details>

Jetzt der ganze Algorithmus. Lies ihn mit den drei Kommentaren zum Aufbau der Tabelle: Wir legen für **jede** Maske eine Zeile mit `n` Werten an (am Anfang `unendlich`). Die Maske 1 (nur Ort 0) hat in Spalte 0 den Wert 0. Dann gehen wir alle Masken **von klein nach groß** durch, probieren für jedes `j` der Maske und jeden noch nicht besuchten Ort `k` die Fahrt `j → k` und merken uns den Besten samt **Vorgänger**. Am Ende gehen wir über die Vorgänger rückwärts und bekommen die Tour. `ZAEHLER_HK` zählt die Schritte.

In [ ]:
ZAEHLER_HK = {'schritte': 0}


def held_karp(d):
    """Exakte kürzeste Rundreise mit Bitmasken. Gibt (Länge, Tour) zurück.
    bester[maske][j] = kürzester Weg, der bei Ort 0 beginnt, genau die Orte der Maske besucht und bei j endet."""
    n = len(d)
    if n == 1:
        return 0, [0]
    unendlich = float('inf')
    bester = []
    vorher = []
    for maske in range(1 << n):
        bester.append([unendlich] * n)
        vorher.append([-1] * n)
    bester[1][0] = 0                              # Maske 1 = nur Ort 0
    for maske in range(1 << n):
        if maske & 1 == 0:                         # Ort 0 ist immer dabei
            continue
        for j in range(n):
            if bester[maske][j] == unendlich:
                continue
            for k in range(n):
                if maske & (1 << k) != 0:          # k schon besucht
                    continue
                ZAEHLER_HK['schritte'] = ZAEHLER_HK['schritte'] + 1
                neue_maske = maske | (1 << k)
                wert = bester[maske][j] + d[j][k]
                if wert < bester[neue_maske][k]:
                    bester[neue_maske][k] = wert
                    vorher[neue_maske][k] = j
    alle = (1 << n) - 1
    ende = 1
    for j in range(2, n):
        if bester[alle][j] + d[j][0] < bester[alle][ende] + d[ende][0]:
            ende = j
    laenge = bester[alle][ende] + d[ende][0]
    tour = []
    maske = alle
    j = ende
    while j != -1:
        tour.append(j)
        vorige = vorher[maske][j]
        maske = maske ^ (1 << j)                   # j aus der Maske nehmen
        j = vorige
    tour.reverse()                                 # beginnt jetzt bei Ort 0
    return laenge, tour

#### Aufgabe 4.3

Prüfe `held_karp` an den Orten mit den Startwerten 1 bis 5 (8 Orte): Vergleiche die Länge mit der Brute-Force-Lösung aus 1.4 (die Zahlen müssen bis auf Rundungsfehler gleich sein), prüfe die Tour mit dem Schiedsrichter und gib an, wie viele Schritte Held-Karp und wie viele Touren Brute Force gebraucht haben. Setze `ZAEHLER_HK['schritte']` vor jedem Aufruf auf 0.

**Erwartete Ausgabe:**
```
1 292.16 292.16 True ok 292.16 1351 2520
2 277.87 277.87 True ok 277.87 1351 2520
3 199.19 199.19 True ok 199.19 1351 2520
4 228.13 228.13 True ok 228.13 1351 2520
5 316.79 316.79 True ok 316.79 1351 2520
```

In [ ]:
for startwert in range(1, 6):
    d = abstandsmatrix(zufalls_orte(startwert, 8))
    ZAEHLER_HK['schritte'] = 0
    # TODO: held_karp aufrufen und mit tour_brute_force vergleichen
    print(startwert)

<details>
<summary>Tipp anzeigen</summary>

`laenge_hk, tour_hk = held_karp(d)` und `laenge_bf, tour_bf, geprueft = tour_brute_force(d)`. Die Länge der Tour, die Held-Karp zurückgibt, rechnest du mit `tour_laenge` nach.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
for startwert in range(1, 6):
    d = abstandsmatrix(zufalls_orte(startwert, 8))
    ZAEHLER_HK['schritte'] = 0
    laenge_hk, tour_hk = held_karp(d)
    laenge_bf, tour_bf, geprueft = tour_brute_force(d)
    print(startwert, round(laenge_hk, 2), round(laenge_bf, 2), abs(laenge_hk - laenge_bf) < 0.000001,
          pruefe_tour(d, tour_hk), round(tour_laenge(d, tour_hk), 2), ZAEHLER_HK['schritte'], geprueft)
```

Beide Verfahren finden dieselbe Länge, und die zurückgegebene Tour hat auch wirklich diese Länge (Gegenprobe!). Held-Karp braucht bei 8 Orten 7 · (6 · 32 + 1) = 1 351 Schritte, Brute Force prüft 2 520 Touren und rechnet für jede 8 Abstände. Bei 8 Orten ist der Unterschied noch klein, die nächste Tabelle zeigt, wie er wächst.

</details>

In [ ]:
print(' n   Schritte Held-Karp   Touren bei Brute Force')
for n in range(6, 14):
    d = abstandsmatrix(zufalls_orte(1, n))
    ZAEHLER_HK['schritte'] = 0
    held_karp(d)
    print(f'{n:2}  {ZAEHLER_HK["schritte"]:>18}  {math.factorial(n - 1) // 2:>22}')

 n   Schritte Held-Karp   Touren bei Brute Force
 6                 165                      60
 7                 486                     360
 8                1351                    2520
 9                3592                   20160
10                9225                  181440
11               23050                 1814400
12               56331                19958400
13              135180               239500800


#### Frage 4.4

1. Lies in der Tabelle ab: Um welchen Faktor wächst die Zahl der Schritte von Held-Karp, wenn man einen Ort **dazunimmt**? Und bei Brute Force?
2. Held-Karp merkt sich eine Tabelle mit `2ⁿ · n` Werten. Wie viele sind das für `n = 13`, für `n = 20` und für `n = 30`? Warum ist `n = 30` mit diesem Verfahren trotzdem unerreichbar, auch wenn die Rechenzeit stimmen würde?
3. Das Problem heißt **NP-schwer**. Was bedeutet das **richtig**, und was bedeutet es **nicht**?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Held-Karp: etwa **Faktor 2,5**, bei größeren `n` immer weniger, er nähert sich 2 (von `n = 12` auf `n = 13`: 56331 auf 135180 Schritte, also Faktor 2,40; die Formel ist `(n − 1)·((n − 2)·2ⁿ⁻³ + 1)`). Brute Force: Faktor **n − 1**, also bei 12 → 13 schon das 12-Fache. Held-Karp wächst auch **exponentiell** (2ⁿ), aber viel langsamer als die Fakultät.
2. `n = 13`: 106 496 Werte, `n = 20`: 20 971 520 (rund 21 Millionen), `n = 30`: 32 212 254 720 (über 32 Milliarden). Eine Python-Liste braucht je Wert mindestens 8 Byte, bei `n = 30` wären das **über 250 Gigabyte**: Der **Speicher** ist die Grenze, lange vor der Zeit. In Python (und im Browser) ist `n ≈ 15` bis `16` das Ende, mit geschickter Programmierung (Zahlen-Felder, C-Code) etwa `n ≈ 25`.
3. **Richtig:** Es ist kein Verfahren bekannt, das **jede** Eingabe in einer Zeit löst, die höchstens wie eine Potenz von `n` wächst (**polynomiell**, zum Beispiel `n³`). Und wer eines fände, könnte damit auch ungezählte andere Probleme in Polynomialzeit lösen (alle Probleme aus der Klasse NP). Das hält man für unmöglich, bewiesen ist es nicht. **Nicht** bedeutet es: Kleine Fälle sind unlösbar (bis `n = 13` rechnet Held-Karp in Sekunden), man braucht keine Heuristiken (doch, gerade deshalb), oder jede **Variante** ist schwer. Die Tour über alle **Straßen** (nächster Abschnitt) ist **nicht** NP-schwer.

</details>

## 5. Wie gut sind die Heuristiken? Schranke, Optimum und Tabelle

Wie in [Notebook 01](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf2/01_pruefen_und_schranken.ipynb) und [Notebook 16](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf2/16_experimente.ipynb) messen wir die Güte auf drei Arten.

**Eine untere Schranke.** Jede Rundreise betritt jeden Ort `i` über eine Kante und verlässt ihn über eine zweite. Diese beiden Kanten sind **mindestens** so lang wie die **zwei kürzesten** Kanten von `i`. Zählt man das für alle Orte zusammen, hat man **jede Kante der Tour zweimal** gezählt (einmal an jedem Ende). Also gilt:

`Länge jeder Tour ≥ (Summe über alle Orte i von (kürzeste + zweitkürzeste Kante von i)) / 2`.

Eine stärkere Schranke bekommt man aus dem **minimalen Spannbaum**: Wirft man aus einer Tour eine Kante weg, bleibt ein Weg durch alle Orte, der ein Spannbaum ist, und der ist mindestens so lang wie der kürzeste Spannbaum. Das wäre eine eigene Aufgabe und gehört zu den vielen Schranken, die man für das TSP kennt.

#### Aufgabe 5.1

Schreibe `schranke_zwei_kuerzeste(d)` und `prozent_ueber(wert, referenz)` (um wie viel Prozent `wert` größer als `referenz` ist). Gib für 10 Orte mit den Startwerten 1 bis 5 aus: das Optimum (Held-Karp), die Schranke und um wie viel Prozent das Optimum über der Schranke liegt.

**Erwartete Ausgabe:**
```
1 302.8 255.1 18.7
2 299.8 263.8 13.6
3 226.3 137.7 64.4
4 263.8 205.9 28.1
5 328.2 279.5 17.4
```

In [ ]:
def schranke_zwei_kuerzeste(d):
    n = len(d)
    # TODO
    return 0


def prozent_ueber(wert, referenz):
    # TODO
    return 0


for startwert in range(1, 6):
    d = abstandsmatrix(zufalls_orte(startwert, 10))
    optimum = held_karp(d)[0]
    schranke = schranke_zwei_kuerzeste(d)
    print(startwert, round(optimum, 1), round(schranke, 1), round(prozent_ueber(optimum, schranke), 1))

<details>
<summary>Tipp anzeigen</summary>

Für jeden Ort `i` sammelst du die Abstände `d[i][j]` zu allen `j != i` in einer Liste, sortierst sie mit `.sort()` und addierst die ersten zwei. Die Gesamtsumme wird durch 2 geteilt.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def schranke_zwei_kuerzeste(d):
    """Untere Schranke: In jeder Tour hat jeder Ort zwei Kanten. Jede ist mindestens so lang wie die zweikürzesten Kanten des Ortes.
    Zählt man das für alle Orte, hat man jede Kante der Tour zweimal gezählt, also teilt man durch 2."""
    n = len(d)
    summe = 0
    for i in range(n):
        abstaende = []
        for j in range(n):
            if j != i:
                abstaende.append(d[i][j])
        abstaende.sort()
        summe = summe + abstaende[0] + abstaende[1]
    return summe / 2


def prozent_ueber(wert, referenz):
    return 100 * (wert - referenz) / referenz


for startwert in range(1, 6):
    d = abstandsmatrix(zufalls_orte(startwert, 10))
    optimum = held_karp(d)[0]
    schranke = schranke_zwei_kuerzeste(d)
    print(startwert, round(optimum, 1), round(schranke, 1), round(prozent_ueber(optimum, schranke), 1))
```

Die Schranke liegt **unter** dem Optimum, und der Abstand ist beträchtlich (die letzte Spalte, bei diesen Startwerten zwischen 13 und 64 Prozent). Eine Tour, die 10 % über der Schranke liegt, ist deshalb **sehr gut**, sie kann dem Optimum sehr nahe sein. Umgekehrt beweist ein Abstand von 30 % zur Schranke noch **nicht**, dass die Tour schlecht ist.

</details>

#### Frage 5.2

1. Warum darf man in der Schranke **zwei** Kanten je Ort nehmen und nicht eine? Was würde passieren, wenn man nur die kürzeste Kante nähme?
2. Ein Schüler behauptet: „Meine Tour ist 8 % über der Schranke, also höchstens 8 % über dem Optimum.“ Stimmt das? Und umgekehrt: Wenn eine Tour 30 % über der Schranke liegt, ist sie dann schlecht?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Jede Tour hat an jedem Ort **zwei** Kanten (hin und weg). Mit nur einer Kante je Ort wäre die Schranke noch **schwächer**, aber auch noch gültig (Summe der kürzesten Kanten, geteilt durch 2 gilt erst recht). Die zwei kürzesten sind besser, weil sie näher an der wahren Tour liegen. Eine zusätzliche Kante je Ort nähme man nicht: Die dritte Kante gehört nicht zwingend zur Tour.
2. Ja: Das Optimum ist **mindestens** so lang wie die Schranke. Die Tour ist also höchstens 8 % über der Schranke, und weil das Optimum nicht darunter liegen kann, ist sie **höchstens 8 % über dem Optimum**. Umgekehrt nein: Eine Tour 30 % über der Schranke kann optimal sein, wenn das Optimum selbst 30 % über der Schranke liegt (wie oben bei einigen Startwerten). Aus einer Schranke kann man also **Gütegarantien** ablesen, aber nur bei **kleinem** Abstand.

</details>

**Tabelle über viele Startwerte.** Jetzt vergleichen wir **sechs Verfahren**: zwei Startverfahren (nächster Nachbar, billigste Einfügung), jeweils ohne Verbesserung, mit 2-opt und mit 2-opt + Or-opt. Gemessen wird der **Abstand zum Optimum in Prozent**, über **zehn** verschiedene Zufallsinstanzen mit je 10 Orten (Startwerte 1 bis 10), als **Mittel** und als **schlechtester Fall**. Wie in [Notebook 04](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf2/04_lokale_suche.ipynb) übergeben wir **Funktionen als Argumente**: Die Verfahren stehen in zwei Wörterbüchern (Name → Funktion), und die Schleife ruft sie über `start(d)` und `verbesserung(d, tour)` auf. So kommt ein siebtes Verfahren mit einer Zeile dazu.

In [ ]:
def start_nn(d):
    return nachster_nachbar(d, 0)


def start_einfuegung(d):
    return billigste_einfuegung(d)


def keine_verbesserung(d, tour):
    return tour


STARTS = {'nächster Nachbar': start_nn, 'billigste Einfügung': start_einfuegung}
VERBESSERUNGEN = {'ohne': keine_verbesserung, '2-opt': zwei_opt, '2-opt + Or-opt': zwei_opt_und_or_opt}


def vergleich(n, startwerte, exakt):
    """Für jede Kombination die Liste der Abstände in Prozent zur Referenz.
    Referenz ist bei exakt=True das Optimum (Held-Karp), sonst die beste gefundene Länge aller sechs Verfahren."""
    ergebnis = {}
    for name_s in STARTS:
        for name_v in VERBESSERUNGEN:
            ergebnis[(name_s, name_v)] = []
    for startwert in startwerte:
        d = abstandsmatrix(zufalls_orte(startwert, n))
        laengen = {}
        for name_s, start in STARTS.items():
            for name_v, verbesserung in VERBESSERUNGEN.items():
                tour = verbesserung(d, start(d))
                assert pruefe_tour(d, tour) == 'ok'
                laengen[(name_s, name_v)] = tour_laenge(d, tour)
        if exakt:
            referenz = held_karp(d)[0]
        else:
            referenz = min(laengen.values())
        for schluessel, laenge in laengen.items():
            ergebnis[schluessel].append(prozent_ueber(laenge, referenz))
    return ergebnis


def zeige_tabelle(ergebnis):
    print(f'{"":22}' + ''.join([f'{name_v:>20}' for name_v in VERBESSERUNGEN]))
    for name_s in STARTS:
        zeile = f'{name_s:22}'
        for name_v in VERBESSERUNGEN:
            werte = ergebnis[(name_s, name_v)]
            zeile = zeile + f'{sum(werte) / len(werte):10.1f} /{max(werte):7.1f}'
        print(zeile)


ergebnis10 = vergleich(10, range(1, 11), True)
print('10 Orte, Startwerte 1 bis 10: Abstand zum Optimum in Prozent (Mittel / schlechtester Fall)')
zeige_tabelle(ergebnis10)

10 Orte, Startwerte 1 bis 10: Abstand zum Optimum in Prozent (Mittel / schlechtester Fall)
                                      ohne               2-opt      2-opt + Or-opt
nächster Nachbar             8.0 /   29.1       0.2 /    1.6       0.0 /    0.0
billigste Einfügung          4.9 /   15.8       0.6 /    4.2       0.0 /    0.0


In [ ]:
ergebnis40 = vergleich(40, range(1, 7), False)
print('40 Orte, Startwerte 1 bis 6: Abstand zur besten gefundenen Tour in Prozent (Mittel / schlechtester Fall)')
zeige_tabelle(ergebnis40)

40 Orte, Startwerte 1 bis 6: Abstand zur besten gefundenen Tour in Prozent (Mittel / schlechtester Fall)
                                      ohne               2-opt      2-opt + Or-opt
nächster Nachbar            26.7 /   36.9       5.3 /   18.0       1.9 /    6.3
billigste Einfügung         12.6 /   19.1       4.9 /   10.5       1.1 /    3.1


#### Frage 5.3

Lies die beiden Tabellen und beantworte:
1. Wie viel bringt die Verbesserung mit 2-opt bei 10 Orten? Und wie oft findet 2-opt + Or-opt in der ersten Tabelle das Optimum?
2. Warum zeigt die zweite Tabelle (40 Orte) die Abstände zur **besten gefundenen** Tour und nicht zum Optimum? Was folgt daraus für die Aussagekraft der Zahlen?
3. Welches Verfahren würdest du für 40 Orte nehmen?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Der nächste Nachbar hat bei 10 Orten im Mittel 8,0 % Abstand, mit 2-opt nur noch 0,2 %; die billigste Einfügung kommt von 4,9 % auf 0,6 %. Die Verbesserung nimmt den größten Teil des Abstands weg. Mit **2-opt + Or-opt** liegt der schlechteste Fall der Tabelle bei 0,0 %: Bei diesen zehn Instanzen findet das Verfahren **immer das Optimum** (Held-Karp bestätigt das). Bei nur zehn Instanzen ist das aber keine Garantie.
2. Held-Karp schafft 40 Orte nicht (Speicher). Als Referenz bleibt die **beste Tour, die einer der sechs Läufe gefunden hat**. Das ist **nicht das Optimum**: Der Abstand zeigt, wie weit ein Verfahren hinter dem **besten der sechs** liegt. Wenn alle sechs weit weg vom Optimum wären, würde die Tabelle das nicht verraten. Deshalb zählt die Tabelle nur **relativ**: Sie ordnet die Verfahren, sagt aber nichts über den absoluten Abstand zum Optimum.
3. Die Verbesserung hilft sehr: nächster Nachbar von 26,7 % auf 5,3 % (2-opt) und 1,9 % (2-opt + Or-opt), billigste Einfügung von 12,6 % auf 4,9 % und 1,1 %. Der Start spielt bei verbesserten Touren eine kleinere Rolle. Die **billigste Einfügung mit 2-opt + Or-opt**, das beste der sechs Verfahren (im Mittel), ist hier die beste Wahl. Der Aufbau kostet O(n³) statt O(n²), das ist bei 40 Orten kein Problem.

</details>

## 6. Touren über alle Straßen: der Eulerkreis

Jetzt die Schwester der Rundreise: Ein **Briefträger** muss in einem Dorf **jede Straße** mindestens einmal abfahren und zum Postamt zurückkehren. Auf den **Orten** kommt es nicht an, sondern auf den **Straßen**. Als Graph: Die Kreuzungen sind Knoten, eine Straße ist eine **Kante** `(u, v, länge)`. Der Schiedsrichter `pruefe_kantentour` (siehe unten) prüft, ob eine Tour nur vorhandene Straßen benutzt, am Ausgangspunkt endet und **jede** Straße mindestens einmal befährt.

Unser Dorf hat acht Kreuzungen in zwei Reihen (die Zahlen an den Straßen sind die Längen):

```
0 --4-- 1 --3-- 2 --5-- 3
|       |       |       |
3       5       2       4
|       |       |       |
4 --4-- 5 --6-- 6 --2-- 7
```

Die beste Lage wäre ein **Eulerkreis**: ein Rundweg, der **jede Straße genau einmal** befährt. Wann gibt es ihn? Das klärt die nächste Frage zuerst, nachdem du den **Grad** einer Kreuzung (die Zahl der Straßen dort) berechnet hast.

#### Aufgabe 6.1

Schreibe `grade(n, kanten)` (eine Liste: Zahl der Straßen an jeder Kreuzung `0` bis `n − 1`) und `ungerade_knoten(n, kanten)` (die Kreuzungen mit **ungeradem** Grad). Eine Straße ist ein Tupel `(u, v, länge)`. Wende beides auf das Dorf an.

**Erwartete Ausgabe:**
```
[2, 3, 3, 2, 2, 3, 3, 2]
[1, 2, 5, 6]
```

In [ ]:
DORF = [(0, 1, 4), (1, 2, 3), (2, 3, 5), (4, 5, 4), (5, 6, 6), (6, 7, 2),
        (0, 4, 3), (1, 5, 5), (2, 6, 2), (3, 7, 4)]


def grade(n, kanten):
    # TODO
    return [0] * n


def ungerade_knoten(n, kanten):
    # TODO
    return []


print(grade(8, DORF))
print(ungerade_knoten(8, DORF))

<details>
<summary>Tipp anzeigen</summary>

Für jede Straße `(u, v, länge)` erhöhst du `g[u]` und `g[v]` um 1. Ungerade ist ein Grad, wenn `grad % 2 == 1`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
DORF = [(0, 1, 4), (1, 2, 3), (2, 3, 5), (4, 5, 4), (5, 6, 6), (6, 7, 2),
        (0, 4, 3), (1, 5, 5), (2, 6, 2), (3, 7, 4)]

def grade(n, kanten):
    """Zahl der Straßen an jeder Kreuzung. Eine Straße ist ein Tupel (u, v, länge)."""
    g = [0] * n
    for u, v, laenge in kanten:
        g[u] = g[u] + 1
        g[v] = g[v] + 1
    return g


def ungerade_knoten(n, kanten):
    g = grade(n, kanten)
    liste = []
    for v in range(n):
        if g[v] % 2 == 1:
            liste.append(v)
    return liste


print(grade(8, DORF))
print(ungerade_knoten(8, DORF))
```

</details>

#### Frage 6.2

1. Warum kann ein Eulerkreis nur existieren, wenn **jede** Kreuzung einen **geraden** Grad hat? (Denke an eine Kreuzung mitten im Rundweg.)
2. Die Bedingung reicht noch nicht ganz. Welche zweite Bedingung braucht man?
3. Im Dorf gibt es vier Kreuzungen mit ungeradem Grad. Kann es auch Dörfer mit genau **einer** ungeraden Kreuzung geben? Begründe mit der Summe aller Grade.

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Jedes Mal, wenn der Rundweg in eine Kreuzung **hineinfährt**, muss er sie auch wieder **verlassen** (außer am Anfang und am Ende, und dort fallen Abfahrt und Ankunft im Rundweg zusammen, also wieder ein Paar). Die Straßen an einer Kreuzung kommen also **paarweise** vor, jede Straße wird nur einmal benutzt: Der Grad ist gerade.
2. Alle Straßen müssen in **einem Stück** liegen (**zusammenhängend**). Zwei getrennte Dreiecke haben nur gerade Grade, aber kein Rundweg verbindet sie. Dann ist keine Tour möglich, die alle Straßen befährt. Es gilt der **Satz von Euler**: Ein zusammenhängender Graph hat genau dann einen Eulerkreis, wenn alle Grade gerade sind.
3. Nein. Jede Straße hat zwei Enden, die Summe aller Grade ist also **doppelt so groß wie die Zahl der Straßen**, also gerade. Eine ungerade Zahl ungerader Summanden ergäbe eine ungerade Summe. Die Zahl der ungeraden Kreuzungen ist deshalb **immer gerade** (0, 2, 4, …).

</details>

**Den Eulerkreis finden: Hierholzer.** Man fährt von der Zentrale los und nimmt immer eine **noch nicht benutzte Straße**, bis man nicht weiter kann (das passiert in einem Graphen mit lauter geraden Graden nur an der **Startkreuzung**). Dabei bleiben vielleicht noch Straßen übrig: Die fährt man später als **Schleifen** ein. Der Trick mit einem **Stapel** erledigt das elegant: Die Liste `stapel` enthält die Fahrt bis jetzt. Kommt man in eine Sackgasse (alle Straßen der Kreuzung benutzt), nimmt man die Kreuzung vom Stapel und hängt sie an das **Ende** der Ergebnisliste. Dann ist man wieder an der vorigen Kreuzung und sucht dort unbenutzte Straßen. Am Ende dreht man die Ergebnisliste um. Jede Straße wird einmal angefasst, der Aufwand ist **O(Straßen)**.

Die Funktion `eulerkreis(n, kanten, start)` macht das. `zusammenhaengend`, `laenge_kantentour` (die Länge einer Fahrt) und der Schiedsrichter `pruefe_kantentour` stehen mit in der Zelle. Gelesen, nicht geschrieben, sie sind kurz und verwenden nur Bekanntes.

In [ ]:
def zusammenhaengend(n, kanten):
    """Sind alle Kreuzungen, an denen eine Straße liegt, von der ersten aus erreichbar?"""
    nachbarn = [[] for v in range(n)]
    for u, v, laenge in kanten:
        nachbarn[u].append(v)
        nachbarn[v].append(u)
    erreicht = {kanten[0][0]}
    stapel = [kanten[0][0]]
    while len(stapel) > 0:
        v = stapel.pop()
        for w in nachbarn[v]:
            if w not in erreicht:
                erreicht.add(w)
                stapel.append(w)
    for v in range(n):
        if len(nachbarn[v]) > 0 and v not in erreicht:
            return False
    return True


def eulerkreis(n, kanten, start):
    """Hierholzer: Kreuzungen eines Rundwegs, der jede Straße genau einmal benutzt (wenn es ihn gibt)."""
    benachbart = [[] for v in range(n)]
    for k in range(len(kanten)):
        u, v, laenge = kanten[k]
        benachbart[u].append((v, k))
        benachbart[v].append((u, k))
    benutzt = [False] * len(kanten)
    stapel = [start]
    kreis = []
    while len(stapel) > 0:
        v = stapel[-1]
        while len(benachbart[v]) > 0 and benutzt[benachbart[v][-1][1]]:
            benachbart[v].pop()                      # diese Straße wurde schon von der anderen Seite befahren
        if len(benachbart[v]) > 0:
            w, k = benachbart[v].pop()
            benutzt[k] = True
            stapel.append(w)                         # weiterfahren
        else:
            kreis.append(stapel.pop())               # Sackgasse: dieser Ort gehört ans Ende des Rundwegs
    kreis.reverse()
    return kreis


def laenge_kantentour(kanten, tour):
    """Länge der Fahrt, wenn man die Straßen zwischen aufeinander folgenden Kreuzungen benutzt (bei Mehrfachstraßen die kürzeste)."""
    kurz = {}
    for u, v, laenge in kanten:
        for paar in ((u, v), (v, u)):
            if paar not in kurz or laenge < kurz[paar]:
                kurz[paar] = laenge
    summe = 0
    for k in range(len(tour) - 1):
        summe = summe + kurz[(tour[k], tour[k + 1])]
    return summe


def pruefe_kantentour(kanten, tour):
    """Schiedsrichter: beginnt und endet die Tour am selben Ort, benutzt sie nur vorhandene Straßen und jede Straße mindestens einmal?"""
    if tour[0] != tour[-1]:
        return 'die Tour endet nicht dort, wo sie beginnt'
    vorhanden = set()
    for u, v, laenge in kanten:
        vorhanden.add((min(u, v), max(u, v)))
    gefahren = set()
    for k in range(len(tour) - 1):
        paar = (min(tour[k], tour[k + 1]), max(tour[k], tour[k + 1]))
        if paar not in vorhanden:
            return f'zwischen {tour[k]} und {tour[k + 1]} gibt es keine Straße'
        gefahren.add(paar)
    for paar in sorted(vorhanden):
        if paar not in gefahren:
            return f'die Straße {paar} wird nie befahren'
    return 'ok'

#### Aufgabe 6.3

Ein Beispiel, bei dem alle Grade gerade sind: ein Dreieck (Kreuzungen 0, 1, 2) und ein Viereck (2, 3, 4, 5), die sich in der Kreuzung 2 berühren. Gib die ungeraden Kreuzungen aus, ob der Plan zusammenhängt, berechne den Eulerkreis ab Kreuzung 0, seine Länge und das Urteil des Schiedsrichters. Ein Eulerkreis hat genau `Zahl der Straßen + 1` Einträge, weil Anfang und Ende dieselbe Kreuzung sind.

**Erwartete Ausgabe:**
```
[]
True
[0, 2, 5, 4, 3, 2, 1, 0] True
17
ok
```

In [ ]:
DREI_VIER = [(0, 1, 2), (1, 2, 3), (2, 0, 4), (2, 3, 1), (3, 4, 2), (4, 5, 3), (5, 2, 2)]
# TODO: ausgeben (1) ungerade Kreuzungen, (2) zusammenhaengend, (3) Eulerkreis, (4) Länge, (5) Urteil des Schiedsrichters

<details>
<summary>Tipp anzeigen</summary>

Du brauchst `ungerade_knoten(6, DREI_VIER)`, `zusammenhaengend(6, DREI_VIER)`, `eulerkreis(6, DREI_VIER, 0)`, `laenge_kantentour(DREI_VIER, kreis)` und `pruefe_kantentour(DREI_VIER, kreis)`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
DREI_VIER = [(0, 1, 2), (1, 2, 3), (2, 0, 4), (2, 3, 1), (3, 4, 2), (4, 5, 3), (5, 2, 2)]
print(ungerade_knoten(6, DREI_VIER))
print(zusammenhaengend(6, DREI_VIER))
kreis = eulerkreis(6, DREI_VIER, 0)
print(kreis, len(kreis) == len(DREI_VIER) + 1)
print(laenge_kantentour(DREI_VIER, kreis))
print(pruefe_kantentour(DREI_VIER, kreis))
```

Es gibt keine ungeraden Kreuzungen, der Plan hängt zusammen, und der Rundweg benutzt jede der 7 Straßen genau einmal, seine Länge ist also die Summe aller Straßenlängen (2 + 3 + 4 + 1 + 2 + 3 + 2 = 17).

</details>

**Finde den Fehler.** Ein Programm prüft nur die Grade und ruft dann `eulerkreis` auf. Hier ein Plan aus **zwei getrennten Dreiecken**. Führe die Zelle aus.

In [ ]:
ZWEI_DREIECKE = [(0, 1, 1), (1, 2, 1), (2, 0, 1), (3, 4, 1), (4, 5, 1), (5, 3, 1)]
print(ungerade_knoten(6, ZWEI_DREIECKE))
kreis = eulerkreis(6, ZWEI_DREIECKE, 0)
print(kreis, len(kreis) == len(ZWEI_DREIECKE) + 1)
print(pruefe_kantentour(ZWEI_DREIECKE, kreis))

[]
[0, 2, 1, 0] False
die Straße (3, 4) wird nie befahren


#### Frage 6.4

Was ist hier schiefgegangen, und wo wäre es ohne den Schiedsrichter unbemerkt geblieben? Was muss man vor dem Aufruf von `eulerkreis` zusätzlich prüfen?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

Alle Grade sind gerade (die Liste der ungeraden Kreuzungen ist leer), aber der Plan besteht aus **zwei Teilen**. `eulerkreis` fährt nur das Dreieck der Startkreuzung ab und gibt `[0, 2, 1, 0]` zurück, einen Rundweg, der **gültig aussieht** (er beginnt und endet bei 0, er benutzt nur vorhandene Straßen), aber die Hälfte der Straßen auslässt. Ohne den Schiedsrichter fällt das nur auf, wenn man **nachzählt**: Der Rundweg hat 4 statt 7 Einträge. Man muss vor dem Aufruf mit `zusammenhaengend` prüfen, dass **alle Straßen von der Startkreuzung aus erreichbar** sind (oder nach dem Aufruf die Länge des Ergebnisses gegen die Zahl der Straßen prüfen).

</details>

**Und wenn es ungerade Kreuzungen gibt?** Dann kann man nicht jede Straße nur einmal fahren. Manche Straßen müssen **zweimal** befahren werden (hin und zurück oder in einer Schleife). Die Idee: **Doppelte Straßen so wählen, dass danach alle Grade gerade sind**, und dann den Eulerkreis des Graphen mit den **verdoppelten** Straßen bilden. Das geht **kürzer**, als man glaubt, denn man verdoppelt nicht einzelne Straßen, sondern **kürzeste Wege** zwischen **Paaren** von ungeraden Kreuzungen: Auf dem Weg von `a` nach `b` ändern alle **inneren** Kreuzungen ihren Grad um 2 (bleiben also gerade oder ungerade), nur `a` und `b` bekommen einen Grad mehr. Man **paart** also die ungeraden Kreuzungen so, dass die **Summe der kürzesten Wege** der Paare möglichst klein ist.

Für die kürzesten Wege nehmen wir **Dijkstra** ([learn_bwinf1, Notebook 07](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/07_dijkstra.ipynb)), das du schon kennst. `alle_abstaende(n, kanten)` ruft es für jede Kreuzung auf und liefert die Abstandstabelle und die Vorgänger, `weg_von_bis(vor, ziel)` liefert aus den Vorgängern den Weg.

In [ ]:
import heapq


def dijkstra(n, kanten, start):
    """Kürzeste Wege von start zu allen Kreuzungen (wie in learn_bwinf1, Notebook 07): Abstände und Vorgänger."""
    nachbarn = [[] for v in range(n)]
    for u, v, laenge in kanten:
        nachbarn[u].append((v, laenge))
        nachbarn[v].append((u, laenge))
    abstand = [float('inf')] * n
    vor = [-1] * n
    abstand[start] = 0
    warteschlange = [(0, start)]
    while len(warteschlange) > 0:
        a, v = heapq.heappop(warteschlange)
        if a > abstand[v]:
            continue
        for w, laenge in nachbarn[v]:
            if a + laenge < abstand[w]:
                abstand[w] = a + laenge
                vor[w] = v
                heapq.heappush(warteschlange, (a + laenge, w))
    return abstand, vor


def weg_von_bis(vor, ziel):
    """Kreuzungen des kürzesten Weges vom Start (aus dem dijkstra den Vorgänger vor berechnet hat) bis ziel."""
    weg = [ziel]
    while vor[weg[-1]] != -1:
        weg.append(vor[weg[-1]])
    weg.reverse()
    return weg


def alle_abstaende(n, kanten):
    """Kürzeste Wege von jeder Kreuzung zu jeder: Abstandsmatrix und für jeden Start die Vorgängerliste."""
    dist = []
    vorgaenger = []
    for s in range(n):
        a, v = dijkstra(n, kanten, s)
        dist.append(a)
        vorgaenger.append(v)
    return dist, vorgaenger

In [ ]:
dist_dorf, vorgaenger_dorf = alle_abstaende(8, DORF)
ung = ungerade_knoten(8, DORF)
for a in ung:
    print(a, [dist_dorf[a][b] for b in ung])
print(weg_von_bis(vorgaenger_dorf[1], 6))

1 [0, 3, 5, 5]
2 [3, 0, 8, 2]
5 [5, 8, 0, 6]
6 [5, 2, 6, 0]
[1, 2, 6]


Die vier Zeilen zeigen die Abstände zwischen den ungeraden Kreuzungen 1, 2, 5, 6 (zum Beispiel ist der kürzeste Weg von 1 nach 6 gleich 5: `1 → 2 → 6` mit 3 + 2). Es gibt drei Möglichkeiten, vier Kreuzungen zu paaren: `(1,2) (5,6)`: 3 + 6 = 9, `(1,5) (2,6)`: 5 + 2 = 7 und `(1,6) (2,5)`: 5 + 8 = 13. Die zweite ist am kürzesten.

Für mehr Kreuzungen probiert `beste_paarung` alle Paarungen durch: Die **erste** Kreuzung der Liste sucht sich einen Partner, mit dem Rest der Liste wird es genauso gemacht (**Rekursion** wie in [Notebook 06]({L2}06_backtracking_gross.ipynb)).

#### Aufgabe 6.5

Schreibe `beste_paarung(knoten, dist)`: Sie gibt `(kleinste Summe, Liste der Paare)` zurück. `knoten` ist die Liste der Kreuzungen, `dist[a][b]` der Abstand. Bei einer leeren Liste ist die Summe 0 und die Paarliste leer. Sonst probierst du für `knoten[0]` jeden Partner `knoten[k]`, paarst den Rest rekursiv und merkst dir die beste Summe. Zähle die Aufrufe in `ZAEHLER_PAARE['aufrufe']` und teste am Dorf.

**Erwartete Ausgabe:**
```
7 [(1, 5), (2, 6)] 7
```

In [ ]:
ZAEHLER_PAARE = {'aufrufe': 0}


def beste_paarung(knoten, dist):
    ZAEHLER_PAARE['aufrufe'] = ZAEHLER_PAARE['aufrufe'] + 1
    # TODO
    return 0, []


summe, paare = beste_paarung(ungerade_knoten(8, DORF), dist_dorf)
print(summe, paare, ZAEHLER_PAARE['aufrufe'])

<details>
<summary>Tipp anzeigen</summary>

Den Rest ohne `knoten[0]` und `knoten[k]` bekommst du mit `knoten[1:k] + knoten[k + 1:]`. Das Ergebnis des Aufrufs mit dem Rest ist `(summe, paare)`. Zur Summe addierst du `dist[a][b]`, und das Paar `(a, b)` kommt zur Liste.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
ZAEHLER_PAARE = {'aufrufe': 0}


def beste_paarung(knoten, dist):
    """Teilt die Kreuzungen in Paare mit kleinster Summe der Wege. Gibt (Summe, Liste der Paare) zurück.
    Der erste Knoten sucht sich einen Partner, mit dem Rest wird es genauso gemacht."""
    ZAEHLER_PAARE['aufrufe'] = ZAEHLER_PAARE['aufrufe'] + 1
    if len(knoten) == 0:
        return 0, []
    a = knoten[0]
    bestes = None
    for k in range(1, len(knoten)):
        b = knoten[k]
        rest = knoten[1:k] + knoten[k + 1:]
        summe, paare = beste_paarung(rest, dist)
        summe = summe + dist[a][b]
        if bestes is None or summe < bestes[0]:
            bestes = (summe, [(a, b)] + paare)
    return bestes


summe, paare = beste_paarung(ungerade_knoten(8, DORF), dist_dorf)
print(summe, paare, ZAEHLER_PAARE['aufrufe'])
```

Die Paarung `(1,5)` und `(2,6)` mit der Summe 7 ist die kürzeste. Die 7 Aufrufe sind 1 für alle vier Kreuzungen, 3 für die drei Reste mit je zwei Kreuzungen und 3 für die leeren Reste.

</details>

In [ ]:
print(' ungerade Kreuzungen   Paarungen   Aufrufe')
for m in range(4, 13, 2):
    zufall = random.Random(1)
    dist_m = []
    for i in range(m):
        dist_m.append([0] * m)
    for i in range(m):
        for j in range(i):
            dist_m[i][j] = dist_m[j][i] = zufall.randint(1, 20)
    ZAEHLER_PAARE['aufrufe'] = 0
    beste_paarung(list(range(m)), dist_m)
    paarungen = 1
    for k in range(m - 1, 0, -2):
        paarungen = paarungen * k
    print(f'{m:20} {paarungen:11} {ZAEHLER_PAARE["aufrufe"]:9}')

 ungerade Kreuzungen   Paarungen   Aufrufe
                   4           3         7
                   6          15        36
                   8         105       253
                  10         945      2278
                  12       10395     25059


#### Frage 6.6

1. Wie viele Paarungen gibt es bei `m` ungeraden Kreuzungen (die Zahl in der Tabelle)? Wie viele bei `m = 20` und `m = 30`? Reicht Durchprobieren für einen Stadtplan mit vielen Kreuzungen?
2. **Gierig:** Man nimmt immer das nächste Paar von noch freien Kreuzungen. Zeige an vier Kreuzungen `a, b, c, d` auf einer Geraden mit den Abständen `a–b = 3`, `b–c = 2`, `c–d = 3` (also `a–c = 5`, `b–d = 5`, `a–d = 8`), dass das nicht immer optimal ist.
3. Wie löst man das für große Pläne? (Nur nennen, nicht programmieren.)

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. `(m − 1) · (m − 3) · … · 3 · 1` Paarungen: 3, 15, 105, 945, 10 395 für `m = 4` bis `12`. Für `m = 20` sind es 654 729 075 (rund 6,5 · 10⁸), für `m = 30` rund 6,2 · 10¹⁵. Bis `m ≈ 16` (rund 2 Millionen) geht Durchprobieren, bei größeren Plänen nicht mehr. Die Aufrufe in der Tabelle wachsen etwa wie die Zahl der Paarungen.
2. Das nächste Paar ist `b–c` mit 2. Übrig bleiben `a` und `d` mit **8**: Summe **10**. Die Paarung `(a,b)` und `(c,d)` kostet nur 3 + 3 = **6**. Gierig nimmt man das kürzeste Paar, aber das verbaut die Partner der beiden Außenkreuzungen.
3. Das ist eine **perfekte Paarung minimalen Gewichts** (in `learn_bwinf1` ist Notebook 14 „Zuordnen“ der Verwandte für **zwei** Gruppen). Für beliebige Graphen gibt es den **Blossom-Algorithmus** von Edmonds mit Aufwand O(m³), in Python zum Beispiel `networkx.min_weight_matching`. Mit Bitmasken (wie bei Held-Karp) geht es bis `m ≈ 20` in `O(2ᵐ · m)`. Für Wettbewerbsaufgaben reicht oft: gierig paaren und dann **zwei Paare tauschen**, falls das kürzer wird (lokale Suche wie in Notebook 04).

</details>

#### Aufgabe 6.7

Setze alles zusammen: `rundweg_alle_strassen(n, kanten, start)` gibt `(Tour, Zusatzlänge, Paare)` zurück. Berechne die ungeraden Kreuzungen und die kürzesten Wege, paare mit `beste_paarung`, **verdopple** die Straßen auf den Wegen der Paare (hänge sie an eine Kopie der Straßenliste an, `weg_von_bis(vorgaenger[a], b)` liefert die Kreuzungen) und bilde den Eulerkreis. Die Länge einer Straße zwischen `u` und `v` steht in einem Wörterbuch `laenge_von[(u, v)]` (für beide Richtungen eintragen).

Teste am Dorf: Gib die Zusatzlänge, die Paare, die Länge des Rundwegs, die Summe der Straßenlängen (`laenge_der_kanten`) und das Urteil des Schiedsrichters aus.

**Erwartete Ausgabe:**
```
7 [(1, 5), (2, 6)]
45 45
[0, 4, 5, 1, 5, 6, 2, 6, 7, 3, 2, 1, 0]
ok
```

In [ ]:
def laenge_der_kanten(kanten):
    summe = 0
    for u, v, laenge in kanten:
        summe = summe + laenge
    return summe


def rundweg_alle_strassen(n, kanten, start):
    # TODO
    return [start], 0, []


tour, zusatz, paare = rundweg_alle_strassen(8, DORF, 0)
print(zusatz, paare)
print(laenge_kantentour(DORF, tour), laenge_der_kanten(DORF) + zusatz)
print(tour)
print(pruefe_kantentour(DORF, tour))

<details>
<summary>Tipp anzeigen</summary>

Die Reihenfolge: (1) `ungerade = ungerade_knoten(...)`, (2) `dist, vorgaenger = alle_abstaende(...)`, (3) `zusatz, paare = beste_paarung(ungerade, dist)`, (4) für jedes Paar `(a, b)` den Weg `weg_von_bis(vorgaenger[a], b)` und für jedes Stück `(weg[k], weg[k + 1])` eine neue Straße anhängen, (5) `eulerkreis(n, doppelt, start)`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def laenge_der_kanten(kanten):
    summe = 0
    for u, v, laenge in kanten:
        summe = summe + laenge
    return summe


def rundweg_alle_strassen(n, kanten, start):
    """Rundweg ab start, der jede Straße mindestens einmal befährt. Gibt (Tour, Zusatzlänge, Paare) zurück."""
    ungerade = ungerade_knoten(n, kanten)
    dist, vorgaenger = alle_abstaende(n, kanten)
    zusatz, paare = beste_paarung(ungerade, dist)
    laenge_von = {}
    for u, v, laenge in kanten:
        laenge_von[(u, v)] = laenge
        laenge_von[(v, u)] = laenge
    doppelt = kanten[:]
    for a, b in paare:
        weg = weg_von_bis(vorgaenger[a], b)
        for k in range(len(weg) - 1):
            doppelt.append((weg[k], weg[k + 1], laenge_von[(weg[k], weg[k + 1])]))
    tour = eulerkreis(n, doppelt, start)
    return tour, zusatz, paare


tour, zusatz, paare = rundweg_alle_strassen(8, DORF, 0)
print(zusatz, paare)
print(laenge_kantentour(DORF, tour), laenge_der_kanten(DORF) + zusatz)
print(tour)
print(pruefe_kantentour(DORF, tour))
```

Die Zusatzlänge ist 7: Die Straßen `1–5` (5) und `2–6` (2) werden zweimal befahren. Der Rundweg ist 38 + 7 = 45 lang, und jede Straße kommt vor. Die Länge ist gleich Summe der Straßen plus Zusatzlänge, **weil der Eulerkreis jede (auch jede doppelte) Straße genau einmal befährt**.

</details>

#### Frage 6.8

1. Warum ist der Rundweg mit **bester** Paarung wirklich der **kürzeste** (wenn der Rundweg keine Fahrt haben darf, die nicht auf dem Plan liegt)? Skizziere den Grund.
2. Das TSP ist NP-schwer. Wie ist es hier? Ist dieses „Briefträgerproblem“ (jede Straße mindestens einmal, ungerichtet) ebenfalls NP-schwer?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Jeder Rundweg, der alle Straßen befährt, benutzt manche Straßen mehrfach. Betrachtet man jede Straße so oft, wie sie befahren wird, hat der entstehende Graph lauter **gerade** Grade (jeder Rundweg ist ein Eulerkreis seines „Fahrtgraphen“). Die **Zusatzstraßen** (alles, was über das Einmal-Befahren hinausgeht) liegen so, dass jede ungerade Kreuzung eine **ungerade** Zahl davon hat und jede gerade Kreuzung eine **gerade**. Man kann sie in **Wege** zerlegen, die ungerade Kreuzungen **paarweise** verbinden. Jeder solche Weg ist mindestens so lang wie der kürzeste Weg zwischen seinen Endpunkten. Also ist die Zusatzlänge mindestens die Summe der kürzesten Wege einer Paarung, und die beste Paarung ist die kleinste.
2. **Nein.** Die Tour über alle Straßen eines ungerichteten Plans ist in **polynomieller Zeit** lösbar: Ein Algorithmus für die minimale perfekte Paarung (Blossom) und der Eulerkreis genügen. Der Unterschied zum TSP ist die Struktur: Beim TSP muss man **Orte** in eine Reihenfolge bringen (n! Möglichkeiten ohne erkennbare Abkürzung). Bei den Straßen ist die Reihenfolge durch den Eulerkreis **egal**, nur die doppelten Straßen sind zu wählen. **Aber:** Gibt es Einbahnstraßen **und** normale Straßen, oder muss man nur **einen Teil** der Straßen befahren und darf die anderen als Verbindung nutzen, wird das Problem wieder **NP-schwer**. Wettbewerbsaufgaben haben oft genau eine solche Nebenbedingung (Müllabfuhr: Aufteilen auf fünf Tage, Ned: begrenzte Ladung).

</details>

## 7. Zum Schluss: die Rundreise mit MiniZinc

Als Gegenprobe zu Held-Karp lösen wir das TSP mit einem **Solver**. Du kennst von den MiniZinc-Arbeitsblättern 2D-Arrays, `sum`, `forall`, Variablen als Index und `alldifferent`. Neu ist die **globale Bedingung `circuit`**:

> `circuit(nach)` verlangt, dass die Pfeile `i → nach[i]` **einen einzigen Kreis durch alle Orte** bilden.

`nach[i]` ist also der Ort, der **direkt nach** Ort `i` besucht wird (statt einer Liste der Orte in der Reihenfolge). Das ist eine **andere Darstellung der Tour** als in Python. `circuit` schließt aus, dass die Pfeile mehrere kleine Kreise bilden (zwei Dreiecke statt einer Rundreise durch sechs Orte), was `alldifferent` allein nicht verhindern würde. Die Länge ist die Summe der Abstände `d[i, nach[i]]`, und wir minimieren sie. Weil MiniZinc nur **ganze Zahlen** rechnet, benutzen wir `ganze_matrix` (auf Zehntel gerundete Abstände), und Held-Karp rechnet bei der Gegenprobe mit **derselben** Tabelle.

Das Modell läuft nicht in JupyterLite. Kopiere es in die MiniZinc IDE oder in den [MiniZinc Playground](https://play.minizinc.org/); dort hast du nur noch die Daten hinzuzufügen. Beim Erzeugen dieses Notebooks wurde es mit **Gecode** gelöst, die Ausgaben stehen darunter.

```minizinc
include "globals.mzn";

int: n;                                     % Zahl der Orte
array[1..n, 1..n] of int: d;                % Abstände (ganze Zahlen)
array[1..n] of var 1..n: nach;              % nach[i] ist der Ort, der direkt nach Ort i besucht wird

constraint circuit(nach);                   % die Pfeile i -> nach[i] bilden genau einen Kreis durch alle Orte

var int: laenge = sum(i in 1..n)(d[i, nach[i]]);
solve minimize laenge;

output ["laenge = \(laenge)\n", "nach = \(nach)\n"];
```

Die Python-Zelle `als_dzn_tsp` erzeugt die Datendatei, hier für nur fünf Orte, damit sie lesbar bleibt:

In [ ]:
def als_dzn_tsp(d):
    """Datendatei (.dzn) für das MiniZinc-Modell: n und die Abstandsmatrix zeilenweise als array2d."""
    n = len(d)
    zahlen = []
    for zeile in d:
        for x in zeile:
            zahlen.append(x)
    return f'n = {n};\nd = array2d(1..{n}, 1..{n}, {zahlen});'


print(als_dzn_tsp(ganze_matrix(zufalls_orte(1, 5))))

n = 5;
d = array2d(1..5, 1..5, [0, 1024, 589, 524, 418, 1024, 0, 654, 953, 656, 589, 654, 0, 877, 515, 524, 953, 877, 0, 375, 418, 656, 515, 375, 0]);


Ergebnis für 10 und 15 Orte (Startwert 1). Die Zeile `laenge` ist die **bewiesen optimale** Länge in Zehntel-Einheiten (am Ende steht `==========`, der Solver hat den Beweis geführt):

**Ausgabe für 10 Orte** (Startwert 1):

```
laenge = 3027
nach = [8, 7, 2, 1, 10, 5, 6, 9, 3, 4]
==========
```

**Ausgabe für 15 Orte** (Startwert 1):

```
laenge = 4097
nach = [12, 7, 13, 10, 1, 11, 14, 9, 3, 5, 4, 8, 15, 6, 2]
==========
```

Die Ausgabe `nach` lässt sich zurück in eine Tour verwandeln, die der Schiedsrichter prüft. Wir folgen von Ort 0 aus den Pfeilen und vergleichen die Länge mit Held-Karp:

In [ ]:
nach_10 = [8, 7, 2, 1, 10, 5, 6, 9, 3, 4]                                   # aus der Ausgabe oben abgeschrieben (Orte ab 1 gezählt)
d10 = ganze_matrix(zufalls_orte(1, 10))
tour = [0]                                         # in Python: Orte ab 0, wir folgen den Pfeilen i -> nach[i]
while len(tour) < 10:
    tour.append(nach_10[tour[-1]] - 1)
print(tour, pruefe_tour(d10, tour))
print(tour_laenge(d10, tour), held_karp(d10)[0])

[0, 7, 8, 2, 1, 6, 5, 4, 9, 3] ok
3027 3027


Für **20 Orte** (Startwert 1) schafft Held-Karp in Python den Speicher nicht mehr (siehe Frage 4.4), MiniZinc dagegen beweist das Optimum. Ausgabe:

```
laenge = 4700
nach = [12, 7, 16, 10, 1, 18, 14, 9, 19, 5, 4, 8, 15, 20, 2, 13, 3, 11, 17, 6]
==========
```

Damit messen wir die sechs Verfahren aus Abschnitt 5 **einmal gegen das echte Optimum**, auf derselben ganzzahligen Tabelle (die Länge ist in Zehntel-Einheiten):

In [ ]:
d20 = ganze_matrix(zufalls_orte(1, 20))
OPTIMUM_MINIZINC = 4700                          # von MiniZinc bewiesen (Ausgabe oben)
print('Optimum (MiniZinc):', OPTIMUM_MINIZINC, ' Schranke:', round(schranke_zwei_kuerzeste(d20)))
for name_s, start in STARTS.items():
    for name_v, verbesserung in VERBESSERUNGEN.items():
        tour = verbesserung(d20, start(d20))
        laenge = tour_laenge(d20, tour)
        print(f'{name_s:22} {name_v:15} {laenge:6} {prozent_ueber(laenge, OPTIMUM_MINIZINC):6.1f} %')

Optimum (MiniZinc): 4700  Schranke: 4008
nächster Nachbar       ohne              5881   25.1 %
nächster Nachbar       2-opt             4827    2.7 %
nächster Nachbar       2-opt + Or-opt    4700    0.0 %
billigste Einfügung    ohne              5465   16.3 %
billigste Einfügung    2-opt             4942    5.1 %
billigste Einfügung    2-opt + Or-opt    4700    0.0 %


Die besten Verfahren liegen bei **0,0 %** über dem Optimum, das schlechteste (der nächste Nachbar ohne Verbesserung) bei **25,1 %**. Das bestätigt in diesem Fall, was die Tabelle für 10 Orte zeigte: Die Verbesserung nimmt fast den ganzen Abstand weg.

**Die Grenze.** Bei 20 Orten beweist Gecode das Optimum in wenigen Sekunden, in einem Bereich, den unser Held-Karp in Python nicht mehr schafft. Mit **25 Orten** (gleicher Startwert) war Gecode nach dem Zeitlimit von 20 Sekunden nicht fertig: Er hatte eine Tour, aber keinen Beweis, und die Tour war weit vom Optimum entfernt; Chuffed lag noch weiter daneben. Der Solver CP-SAT schaffte 25 Orte (bei diesem Startwert), nicht mehr aber 30 und 40 Orte. Für größere Fälle braucht man also die Heuristiken aus den Abschnitten 2 und 3, und MiniZinc dient als **Gegenprobe** für kleine Fälle, so wie hier.

#### Frage 7.1

Warum eignet sich MiniZinc hier gut als **Gegenprobe**, aber schlecht für die Rundreise mit 200 Orten? Was kann der Solver, das die Heuristiken **nicht** können?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

Der Solver **beweist** die Optimalität: Am Ende steht `==========`, er hat alle anderen Touren ausgeschlossen (durch kluges Abschneiden, nicht durch Durchprobieren). So eine Aussage kann keine Heuristik machen, die nur eine gute Tour findet. Die Beweisarbeit wächst aber **exponentiell** mit `n`. Bei 200 Orten wäre schon die Beschreibung riesig (40 000 Abstände), und der Solver findet zwar mit etwas Glück gute Touren, aber **ohne Beweis**. Für solche Größen sind 2-opt und Or-opt mit mehreren Startwerten besser.

</details>

## Wettbewerbs-Ecke 1: Gießroboter bauen ihre Routen

*Angelehnt an Aufgabe 2 „Gießroboter“ der 2. Runde des 44. Bundeswettbewerbs Informatik (2025/26).*

Ein botanischer Garten will seine Bäume mit **möglichst wenigen** Gießrobotern bewässern. Jeder Roboter fährt eine **Rundroute** durch einige Bäume (Start und Ende an einem Baum, seiner Ladestation), und jede Route darf höchstens `lmax` lang sein. Jeder Baum muss auf einer Route liegen. Die Bäume sind Punkte `(x, y)` und die Routen Rundreisen im Sinne dieses Notebooks.

In [Notebook 16](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf2/16_experimente.ipynb) war das Routenverfahren **gegeben** und wurde nur im Experiment verglichen. Hier **baust du es selbst**: Eine Route entsteht durch **billigste Einfügung** (Umweg am kleinsten) und wird nach jedem Einfügen mit **2-opt** verkürzt, solange sie höchstens `lmax` lang bleibt. Dann beginnt die nächste Route bei dem Baum, der **am weitesten von der Mitte** des Gartens entfernt und noch nicht bewässert ist. Am Ende vergleichst du die Routen mit der **exakten** Rundreise von Held-Karp. Das Lösungsnotebook [Gießroboter](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/44_Giessroboter/giessroboter_loesung.ipynb) ist ausführlicher (große Gärten mit einem Gitter, Routen auflösen, mehrere Gewichte); hier sind die Gärten klein: die Dateien `daten/giessroboter01.txt`, `giessroboter02.txt` und `giessroboter04_40.txt` (die ganzen Beispiele `roboter01` und `roboter02` mit 13 und 18 Bäumen) und `giessroboter04_40.txt`, `giessroboter04_80.txt`, `giessroboter05_60.txt` (die ersten 40, 80 und 60 Bäume von `roboter04` und `roboter05`; `lmax` steht jeweils in der ersten Zeile).

In [ ]:
def lies_garten(pfad):
    """Erste Zeile lmax, zweite Zeile Zahl der Bäume, dann pro Baum 'Nummer x y'. Gibt (lmax, Liste der Punkte) zurück."""
    zeilen = open(pfad).read().split('\n')
    lmax = int(zeilen[0])
    n = int(zeilen[1])
    baeume = []
    for k in range(n):
        teile = zeilen[2 + k].split()
        baeume.append((int(teile[1]), int(teile[2])))
    return lmax, baeume


lmax01, baeume01 = lies_garten('daten/giessroboter01.txt')
print(lmax01, len(baeume01), baeume01[:3])

#### Aufgabe E1.1

Schreibe `baue_route(d, lmax, start, frei, mit_zwei_opt)`. Die Route ist eine Liste von Baumnummern, sie beginnt bei `start`. Solange es freie Bäume gibt: Suche unter allen freien Bäumen `c` und allen Stellen `k` der Route (zwischen `route[k]` und `route[(k + 1) % len(route)]`) den **kleinsten Umweg**. Passt er noch (`laenge + umweg ≤ lmax`, mit einer winzigen Toleranz `0.000001`), füge den Baum ein, setze `frei[c] = False` und kürze die Route mit `zwei_opt`, falls `mit_zwei_opt` gilt. Sonst bricht die Route ab. Teste an `giessroboter01.txt` ab Baum 0 (Nummern ab 0).

**Erwartete Ausgabe:**
```
[0, 1, 5, 2, 3] 169.6 175
```

In [ ]:
def baue_route(d, lmax, start, frei, mit_zwei_opt):
    route = [start]
    frei[start] = False
    # TODO
    return route


d01 = abstandsmatrix(baeume01)
frei = [True] * len(baeume01)
route = baue_route(d01, lmax01, 0, frei, True)
print(route, round(tour_laenge(d01, route), 1), lmax01)

<details>
<summary>Tipp anzeigen</summary>

Der Umweg beim Einfügen von `c` zwischen `a` und `b` ist `d[a][c] + d[c][b] - d[a][b]` (bei einer Route aus einem einzigen Baum ist `a == b`, das gibt `2 * d[a][c]`, und das stimmt auch). Die beste Stelle merkst du dir als Tupel `(umweg, c, k + 1)`, `route.insert(k + 1, c)` fügt ein.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def baue_route(d, lmax, start, frei, mit_zwei_opt):
    """Eine Route ab start: immer den freien Baum mit dem kleinsten Umweg einfügen, solange die Route höchstens lmax lang bleibt.
    frei[i] ist True, solange Baum i noch auf keiner Route liegt (wird hier auf False gesetzt)."""
    route = [start]
    frei[start] = False
    laenge = 0.0
    while True:
        beste = None
        for c in range(len(d)):
            if frei[c]:
                for k in range(len(route)):
                    a = route[k]
                    b = route[(k + 1) % len(route)]
                    umweg = d[a][c] + d[c][b] - d[a][b]
                    if beste is None or umweg < beste[0]:
                        beste = (umweg, c, k + 1)
        if beste is None or laenge + beste[0] > lmax - 0.000001:
            break
        umweg, c, position = beste
        route.insert(position, c)
        frei[c] = False
        if mit_zwei_opt:
            route = zwei_opt(d, route)
        laenge = tour_laenge(d, route)
    return route


d01 = abstandsmatrix(baeume01)
frei = [True] * len(baeume01)
route = baue_route(d01, lmax01, 0, frei, True)
print(route, round(tour_laenge(d01, route), 1), lmax01)
```

Die erste Route fasst fünf Bäume und ist knapp unter `lmax = 175` lang, das nächste Einfügen hätte sie über die Grenze gebracht. Die Route ist eine Rundreise, die Zählung beginnt bei Baum 0, das Ergebnis ist eine Liste von Baumnummern.

</details>

Jetzt alle Routen: `baue_alle_routen` sortiert die Bäume **von außen nach innen** (nach dem Abstand zum Mittelpunkt aller Bäume, `key=lambda i: …` gibt die Sortierregel als kleine Funktion an) und beginnt bei jedem noch freien Baum eine neue Route. `pruefe_routen` ist der **Schiedsrichter**: Jeder Baum liegt auf einer Route, und jede Route ist höchstens `lmax` lang.

In [ ]:
def baue_alle_routen(baeume, lmax, mit_zwei_opt):
    d = abstandsmatrix(baeume)
    n = len(baeume)
    mitte = (sum([b[0] for b in baeume]) / n, sum([b[1] for b in baeume]) / n)
    reihenfolge = sorted(range(n), key=lambda i: -math.dist(baeume[i], mitte))      # von außen nach innen
    frei = [True] * n
    routen = []
    for start in reihenfolge:
        if frei[start]:
            routen.append(baue_route(d, lmax, start, frei, mit_zwei_opt))
    return d, routen


def pruefe_routen(d, lmax, routen):
    gewaessert = [False] * len(d)
    for route in routen:
        if tour_laenge(d, route) > lmax:
            return 'eine Route ist zu lang'
        for baum in route:
            gewaessert[baum] = True
    if False in gewaessert:
        return 'ein Baum wird nicht bewässert'
    return 'ok'

#### Aufgabe E1.2

Wie viele Roboter brauchen die Gärten **ohne** und **mit** 2-opt, und wie lang sind alle Routen zusammen? Die Zelle ist fertig, führe sie aus und lies die Ausgabe. Die letzte Zeile ist der Garten `giessroboter05_60.txt` mit der kleineren Maximallänge 400 statt 1 400. Zum Vergleich: Das Lösungsnotebook [Gießroboter](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/44_Giessroboter/giessroboter_loesung.ipynb) und das Experiment in Notebook 16 kommen für die ersten drei Dateien auf 3, 8 und 9 Roboter.

**Erwartete Ausgabe:**
```
Datei                   Bäume  lmax  Roboter ohne/mit 2-opt  Gesamtlänge ohne/mit 2-opt  Schiedsrichter
giessroboter01.txt         13   175          3 /  3          461.2 /    461.2  ok/ok
giessroboter02.txt         18   175          8 /  8          738.6 /    738.6  ok/ok
giessroboter04_40.txt      40   100          9 /  9          502.2 /    502.2  ok/ok
giessroboter05_60.txt      60  1400          2 /  2          978.0 /    935.2  ok/ok
giessroboter05_60.txt      60   400          4 /  4         1005.3 /    822.7  ok/ok
```

In [ ]:
faelle = [('giessroboter01.txt', None), ('giessroboter02.txt', None), ('giessroboter04_40.txt', None),
          ('giessroboter05_60.txt', None), ('giessroboter05_60.txt', 400)]
print('Datei                   Bäume  lmax  Roboter ohne/mit 2-opt  Gesamtlänge ohne/mit 2-opt  Schiedsrichter')
for datei, neues_lmax in faelle:
    lmax, baeume = lies_garten('daten/' + datei)
    if neues_lmax is not None:
        lmax = neues_lmax
    d, routen_ohne = baue_alle_routen(baeume, lmax, False)
    d, routen_mit = baue_alle_routen(baeume, lmax, True)
    ganz_ohne = sum([tour_laenge(d, r) for r in routen_ohne])
    ganz_mit = sum([tour_laenge(d, r) for r in routen_mit])
    urteil = pruefe_routen(d, lmax, routen_ohne) + '/' + pruefe_routen(d, lmax, routen_mit)
    print(f'{datei:22} {len(baeume):6} {lmax:5} {len(routen_ohne):10} /{len(routen_mit):3} {ganz_ohne:14.1f} /{ganz_mit:9.1f}  {urteil}')

<details>
<summary>Tipp anzeigen</summary>

Es genügt, die Zelle auszuführen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
faelle = [('giessroboter01.txt', None), ('giessroboter02.txt', None), ('giessroboter04_40.txt', None),
          ('giessroboter05_60.txt', None), ('giessroboter05_60.txt', 400)]
print('Datei                   Bäume  lmax  Roboter ohne/mit 2-opt  Gesamtlänge ohne/mit 2-opt  Schiedsrichter')
for datei, neues_lmax in faelle:
    lmax, baeume = lies_garten('daten/' + datei)
    if neues_lmax is not None:
        lmax = neues_lmax
    d, routen_ohne = baue_alle_routen(baeume, lmax, False)
    d, routen_mit = baue_alle_routen(baeume, lmax, True)
    ganz_ohne = sum([tour_laenge(d, r) for r in routen_ohne])
    ganz_mit = sum([tour_laenge(d, r) for r in routen_mit])
    urteil = pruefe_routen(d, lmax, routen_ohne) + '/' + pruefe_routen(d, lmax, routen_mit)
    print(f'{datei:22} {len(baeume):6} {lmax:5} {len(routen_ohne):10} /{len(routen_mit):3} {ganz_ohne:14.1f} /{ganz_mit:9.1f}  {urteil}')
```

</details>

#### Aufgabe E1.3

Jetzt der Vergleich mit der **exakten** Rundreise. Für jede Route rechnet Held-Karp die kürzeste Rundreise **durch dieselben Bäume** (das geht, weil die Routen dieser Gärten höchstens 10 Bäume haben). Die Tabelle zeigt die Summe der Routenlängen ohne 2-opt, mit 2-opt und exakt, und in wie vielen Routen 2-opt nicht das Exakte findet. Die Zelle ist fertig, führe sie aus.

**Erwartete Ausgabe:**
```
Datei                  Routen  größte Route  Länge ohne 2-opt  mit 2-opt  exakt  Routen mit 2-opt nicht exakt
giessroboter01.txt          3             5             461.2      461.2  461.2     0
giessroboter02.txt          8             4             738.6      738.6  738.6     0
giessroboter04_40.txt       9            10             502.2      502.2  502.2     0
giessroboter04_80.txt      16            10            1246.6     1246.6 1246.6     0
```

In [ ]:
dateien = ['giessroboter01.txt', 'giessroboter02.txt', 'giessroboter04_40.txt', 'giessroboter04_80.txt']
print('Datei                  Routen  größte Route  Länge ohne 2-opt  mit 2-opt  exakt  Routen mit 2-opt nicht exakt')
for datei in dateien:
    lmax, baeume = lies_garten('daten/' + datei)
    d, routen_ohne = baue_alle_routen(baeume, lmax, False)
    d, routen_mit = baue_alle_routen(baeume, lmax, True)
    summe_ohne = 0
    summe_mit = 0
    summe_exakt = 0
    nicht_exakt = 0
    for route in routen_mit:
        teil = [[d[i][j] for j in route] for i in route]       # Abstände nur zwischen den Bäumen dieser Route
        exakt = held_karp(teil)[0]
        summe_exakt = summe_exakt + exakt
        summe_mit = summe_mit + tour_laenge(d, route)
        if tour_laenge(d, route) > exakt + 0.000001:
            nicht_exakt = nicht_exakt + 1
    for route in routen_ohne:
        summe_ohne = summe_ohne + tour_laenge(d, route)
    groesste = max([len(r) for r in routen_mit])
    print(f'{datei:22} {len(routen_mit):6} {groesste:13} {summe_ohne:17.1f} {summe_mit:10.1f} {summe_exakt:6.1f} {nicht_exakt:5}')

<details>
<summary>Tipp anzeigen</summary>

Es genügt, die Zelle auszuführen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
dateien = ['giessroboter01.txt', 'giessroboter02.txt', 'giessroboter04_40.txt', 'giessroboter04_80.txt']
print('Datei                  Routen  größte Route  Länge ohne 2-opt  mit 2-opt  exakt  Routen mit 2-opt nicht exakt')
for datei in dateien:
    lmax, baeume = lies_garten('daten/' + datei)
    d, routen_ohne = baue_alle_routen(baeume, lmax, False)
    d, routen_mit = baue_alle_routen(baeume, lmax, True)
    summe_ohne = 0
    summe_mit = 0
    summe_exakt = 0
    nicht_exakt = 0
    for route in routen_mit:
        teil = [[d[i][j] for j in route] for i in route]       # Abstände nur zwischen den Bäumen dieser Route
        exakt = held_karp(teil)[0]
        summe_exakt = summe_exakt + exakt
        summe_mit = summe_mit + tour_laenge(d, route)
        if tour_laenge(d, route) > exakt + 0.000001:
            nicht_exakt = nicht_exakt + 1
    for route in routen_ohne:
        summe_ohne = summe_ohne + tour_laenge(d, route)
    groesste = max([len(r) for r in routen_mit])
    print(f'{datei:22} {len(routen_mit):6} {groesste:13} {summe_ohne:17.1f} {summe_mit:10.1f} {summe_exakt:6.1f} {nicht_exakt:5}')
```

</details>

#### Frage E1.4

1. Was zeigen die beiden Tabellen über die Wirkung von 2-opt auf die **Länge** der Routen und auf die **Zahl der Roboter**?
2. In E1.3 ist 2-opt nie schlechter als die exakte Lösung. Warum kann man daraus **nicht** schließen, dass 2-opt immer optimal ist?
3. Wie könnte man mit weniger Robotern auskommen, wenn nicht durch bessere Routen?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. 2-opt macht lange Routen deutlich kürzer: Im Garten `giessroboter05_60.txt` (Routen mit 26 und 34 Bäumen) sinkt die Gesamtlänge von 978.0 auf 935.2, bei `lmax = 400` von 1005.3 auf 822.7 (rund 18 %). Bei den kleinen Gärten mit Routen aus höchstens 10 Bäumen ändert sich **nichts**: Die billigste Einfügung baut dort schon die beste Route. Die **Zahl der Roboter** bleibt in allen Fällen gleich. Der gewonnene Platz reicht in diesen Beispielen nicht, um einen Roboter zu sparen.
2. Es sind nur **wenige kleine** Routen (höchstens 10 Bäume). Bei kleinen Routen ist die Einfügung selbst fast immer schon gut, und 2-opt dreht nur die Kreuzungen weg. Bei 34 Bäumen kann man das Optimum nicht berechnen (Held-Karp reicht nur bis etwa 13 Bäume), also weiß man nur, dass 2-opt dort **höchstens** so schlecht wie bei kleinen Routen sein kann. Das ist ein **Hinweis**, kein Beweis. Das Notebook 04 hat gezeigt, dass 2-opt in einem lokalen Optimum stecken bleiben kann.
3. Durch eine **bessere Verteilung der Bäume auf die Routen**, nicht durch bessere einzelne Routen: Man löst eine Route auf und fügt ihre Bäume in die übrigen ein (das macht das Lösungsnotebook; bei den großen Gärten braucht sein einfaches Verfahren laut dem Vergleich dort etwa 2 bis 7 Prozent mehr Roboter als die Lösungshinweise). Außerdem wählt man den **Startbaum** einer neuen Route klug (hier: von außen nach innen).

</details>

## Wettbewerbs-Ecke 2: Müllabfuhr

*Angelehnt an Aufgabe 1 „Müllabfuhr“ der 2. Runde des 40. Bundeswettbewerbs Informatik (2021/22).*

Die Müllabfuhr hat **ein** Müllauto. Jede Straße muss **mindestens einmal pro Woche** befahren werden, jede Tagesroute beginnt und endet in der Zentrale, und die **längste Tagesstrecke** soll möglichst kurz sein. Gewendet wird nur an Kreuzungen. Der Plan hat `n` Kreuzungen (die Zentrale ist Kreuzung 0) und Straßen `u v länge`.

Wir **vereinfachen**: Zuerst nur **eine Tour für alle Straßen** (statt fünf Tagen), also genau der Rundweg dieses Notebooks. Aus dem Rundweg folgt eine **untere Schranke für die längste Tagesstrecke**, und mit der vergleichen wir. Das **Aufteilen** des Rundwegs in fünf Tage ist der schwierige Teil der Aufgabe, ihn macht das Lösungsnotebook [Müllabfuhr](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/40_Muellabfuhr/muellabfuhr_loesung.ipynb) (mit besserer Paarung, vielen Rundwegen und MiniZinc).

**Die Schranke.** Jede Wochenlösung besteht aus fünf Rundwegen ab der Zentrale. Hängt man sie aneinander, entsteht **ein** Rundweg, der alle Straßen befährt, er ist also mindestens so lang wie der **kürzeste** Rundweg `R` (unser `Rundweg`). Der längste Tag ist deshalb mindestens `R / 5`, aufgerundet. Außerdem muss das Auto an einem Tag die **entfernteste Straße** erreichen und zurückkehren: Für eine Straße `(u, v)` sind das mindestens `d0[u] + länge + d0[v]`, wobei `d0` die kürzesten Wege von der Zentrale sind (Dijkstra). Die größere der beiden Zahlen ist die Schranke.

Die Dateien `daten/muellabfuhr0.txt`, `muellabfuhr1.txt` und `muellabfuhr2.txt` sind die ersten drei echten Beispiele (10, 8 und 15 Kreuzungen).

In [ ]:
def lies_plan(pfad):
    """Erste Zeile: Zahl der Kreuzungen und der Straßen, dann pro Straße 'u v länge'."""
    zeilen = open(pfad).read().split('\n')
    n, m = [int(x) for x in zeilen[0].split()]
    kanten = []
    for k in range(m):
        u, v, laenge = [int(x) for x in zeilen[1 + k].split()]
        kanten.append((u, v, laenge))
    return n, kanten


n0, kanten0 = lies_plan('daten/muellabfuhr0.txt')
print(n0, len(kanten0), kanten0[:3])

#### Aufgabe E2.1

Schreibe `schranke_tag(n, kanten, rundweg, tage)` nach der Beschreibung oben. Mit `math.ceil` rundest du auf, die kürzesten Wege von der Zentrale liefert `dijkstra(n, kanten, 0)` (als `(abstand, vorgaenger)`). Teste am Plan `muellabfuhr1.txt`.

**Erwartete Ausgabe:**
```
66 18
```

In [ ]:
def schranke_tag(n, kanten, rundweg, tage):
    # TODO: das Größere von (Rundweg / tage aufgerundet) und (d0[u] + länge + d0[v] für die entfernteste Straße)
    return 0


n1, kanten1 = lies_plan('daten/muellabfuhr1.txt')
tour1, zusatz1, paare1 = rundweg_alle_strassen(n1, kanten1, 0)
rundweg1 = laenge_kantentour(kanten1, tour1)
print(rundweg1, schranke_tag(n1, kanten1, rundweg1, 5))

<details>
<summary>Tipp anzeigen</summary>

`d0[u] + laenge + d0[v]` ist für jede Straße der Weg von der Zentrale zu `u`, über die Straße zu `v` und von dort zurück. Die Schranke ist das Maximum über alle Straßen und den Wert `math.ceil(rundweg / tage)`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def schranke_tag(n, kanten, rundweg, tage):
    d0, vor = dijkstra(n, kanten, 0)
    schranke = math.ceil(rundweg / tage)
    for u, v, laenge in kanten:
        schranke = max(schranke, d0[u] + laenge + d0[v])
    return schranke


n1, kanten1 = lies_plan('daten/muellabfuhr1.txt')
tour1, zusatz1, paare1 = rundweg_alle_strassen(n1, kanten1, 0)
rundweg1 = laenge_kantentour(kanten1, tour1)
print(rundweg1, schranke_tag(n1, kanten1, rundweg1, 5))
```

Der Rundweg ist 66 lang, ein Fünftel aufgerundet sind 14. Es gibt aber eine Straße, zu der man schon 18 braucht (hin, Straße, zurück): Hier bestimmt die **entfernteste Straße** die Schranke, nicht die Gesamtlänge.

</details>

#### Aufgabe E2.2

Wende `rundweg_alle_strassen` auf die drei Pläne an und gib je Plan aus: Zahl der Kreuzungen und Straßen, Zahl der ungeraden Kreuzungen, Summe der Straßenlängen, Zusatzlänge, Länge des Rundwegs, Urteil des Schiedsrichters und die Schranke für die längste Tagesstrecke bei fünf Tagen. Die Zelle ist fertig, führe sie aus.

**Erwartete Ausgabe:**
```
Plan              Kreuzungen  Straßen  ungerade  Summe  Zusatz  Rundweg  Schiedsrichter  Schranke Tag
muellabfuhr0.txt         10       13         4     13       5       18  ok                        4
muellabfuhr1.txt          8       13         4     57       9       66  ok                       18
muellabfuhr2.txt         15       34        10     34       5       39  ok                        8
```

In [ ]:
print('Plan              Kreuzungen  Straßen  ungerade  Summe  Zusatz  Rundweg  Schiedsrichter  Schranke Tag')
for nr in range(3):
    n, kanten = lies_plan('daten/muellabfuhr' + str(nr) + '.txt')
    tour, zusatz, paare = rundweg_alle_strassen(n, kanten, 0)
    rundweg = laenge_kantentour(kanten, tour)
    print(f'muellabfuhr{nr}.txt {n:10} {len(kanten):8} {len(ungerade_knoten(n, kanten)):9} {laenge_der_kanten(kanten):6} '
          f'{zusatz:7} {rundweg:8}  {pruefe_kantentour(kanten, tour):14} {schranke_tag(n, kanten, rundweg, 5):12}')

<details>
<summary>Tipp anzeigen</summary>

Es genügt, die Zelle auszuführen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
print('Plan              Kreuzungen  Straßen  ungerade  Summe  Zusatz  Rundweg  Schiedsrichter  Schranke Tag')
for nr in range(3):
    n, kanten = lies_plan('daten/muellabfuhr' + str(nr) + '.txt')
    tour, zusatz, paare = rundweg_alle_strassen(n, kanten, 0)
    rundweg = laenge_kantentour(kanten, tour)
    print(f'muellabfuhr{nr}.txt {n:10} {len(kanten):8} {len(ungerade_knoten(n, kanten)):9} {laenge_der_kanten(kanten):6} '
          f'{zusatz:7} {rundweg:8}  {pruefe_kantentour(kanten, tour):14} {schranke_tag(n, kanten, rundweg, 5):12}')
```

</details>

#### Frage E2.3

1. Die Schranke ist bei `muellabfuhr0` genau 4. Die Lösungshinweise geben als **optimale** längste Tagesstrecke für die drei Pläne 4, 18 und 9 an. Was bedeutet das für unsere Schranken 4, 18 und 8? Wieso ist die Schranke beim dritten Plan nicht erreicht?
2. Warum ist die Zahl `Rundweg / 5` eine Schranke, obwohl die fünf Tage ja **nicht** zu einem einzigen Rundweg gehören?
3. Welcher Teil der Aufgabe ist „leicht“ (in Polynomialzeit lösbar) und welcher schwer?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Bei den ersten beiden Plänen **trifft die Schranke das Optimum**: 4 und 18. Die Lösung des Lösungsnotebooks erreicht beide. Beim dritten Plan liegt das Optimum bei 9, unsere Schranke bei 8: Die Schranke ist **nicht scharf**. Ein Fünftel des Rundwegs (39 / 5 = 7,8, aufgerundet 8) wäre nur erreichbar, wenn man den Rundweg an **genau passenden** Stellen in fünf gleich lange Stücke schneiden könnte und jedes Stück **zur Zentrale zurück** nichts kostet. Der Weg von der Zentrale zum Anfang und zurück vom Ende kostet aber extra.
2. Hängt man die fünf Tagesrouten aneinander, entsteht **ein** Rundweg ab der Zentrale, der alle Straßen befährt. Er ist mindestens so lang wie der kürzeste solche Rundweg `R`. Die fünf Tage sind zusammen also mindestens `R` lang, und der längste Tag ist mindestens ein Fünftel davon.
3. Der Rundweg über alle Straßen ist in **Polynomialzeit** lösbar (Abschnitt 6). Das **Aufteilen** auf fünf Tage ist schwer: Verteilt man die Straßen so, dass die längste Tagesstrecke möglichst klein wird, ist das wie das Aufteilen von Paketen auf Lieferwagen (Notebook 01) und **NP-schwer**. Man nimmt hier Heuristiken (Rundweg in Stücke teilen, Tauschen) oder ein Modell für einen Solver.

</details>

## Wettbewerbs-Ecke 3: Weniger krumme Touren (für Schnelle)

*Angelehnt an Aufgabe 1 „Weniger krumme Touren“ der 2. Runde des 41. Bundeswettbewerbs Informatik (2022/23).*

Anton muss alle Außenstellen in Australien abfliegen, darf aber an **keinem** Ort mit einem Winkel von **mehr als 90 Grad** abbiegen. Die Route ist eine **Strecke durch alle Orte** (ein Weg, keine Rundreise; Anfang und Ende sind frei), geradlinig von Ort zu Ort, und möglichst kurz. Das ist die Rundreise mit einer **Nebenbedingung**, die jede Verbesserung erst prüfen muss, und deshalb gut als Übung für Schnelle: Die Straßen-Variante dieses Notebooks wäre die Aufgabe „Ned“ (Streuen), sie wiederholt aber im Wesentlichen die Müllabfuhr.

**Abbiegung prüfen.** Fliegt man von `a` nach `b` und weiter nach `c`, ist der Abbiegewinkel in `b` (das ist 180° minus der Innenwinkel des Dreiecks `a b c` bei `b`) genau dann höchstens 90°, wenn der Innenwinkel mindestens 90° ist, also wenn das **Skalarprodukt** der Vektoren `b → a` und `b → c` höchstens 0 ist. Es braucht keinen Winkel und keine Wurzel: `(a[0] − b[0]) · (c[0] − b[0]) + (a[1] − b[1]) · (c[1] − b[1]) ≤ 0`.

Die Datei `daten/wenigerkrumm4.txt` ist das vierte echte Beispiel (25 Orte, eine Zeile `x y` pro Ort; sie hat Kommazahlen und negative Koordinaten).

In [ ]:
def lies_orte(pfad):
    orte = []
    for zeile in open(pfad).read().split('\n'):
        if zeile.strip() != '':
            x, y = zeile.split()
            orte.append((float(x), float(y)))
    return orte


orte4 = lies_orte('daten/wenigerkrumm4.txt')
print(len(orte4), orte4[:2])

#### Aufgabe E3.1

Schreibe `erlaubt(a, b, c)` (Skalarprodukt höchstens 0, mit der Toleranz `0.000000001`) und `ist_gueltig(route)` (alle drei aufeinander folgenden Punkte erlaubt). Teste: Von `(0, 0)` nach `(10, 0)` weiter nach `(20, 0)` (geradeaus), `(10, 10)` (genau 90°), `(0, 5)` und `(3, 1)` (spitze Kehrtwenden).

**Erwartete Ausgabe:**
```
(20, 0) True
(10, 10) True
(0, 5) False
(3, 1) False
True False
```

In [ ]:
def erlaubt(a, b, c):
    # TODO
    return True


def ist_gueltig(route):
    # TODO
    return True

a, b = (0, 0), (10, 0)
for c in [(20, 0), (10, 10), (0, 5), (3, 1)]:
    print(c, erlaubt(a, b, c))
print(ist_gueltig([(0, 0), (10, 0), (10, 10), (0, 10)]), ist_gueltig([(0, 0), (10, 0), (0, 5)]))

<details>
<summary>Tipp anzeigen</summary>

Das Skalarprodukt von `b → a` und `b → c` ist `(a[0] - b[0]) * (c[0] - b[0]) + (a[1] - b[1]) * (c[1] - b[1])`. Für `ist_gueltig` läufst du `i` von 0 bis `len(route) - 3` und prüfst `route[i], route[i + 1], route[i + 2]`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def erlaubt(a, b, c):
    """Darf man von a über b nach c fliegen (Abbiegewinkel in b höchstens 90 Grad)? Mit winziger Toleranz für Kommazahlen."""
    skalarprodukt = (a[0] - b[0]) * (c[0] - b[0]) + (a[1] - b[1]) * (c[1] - b[1])
    return skalarprodukt <= 0.000000001


def ist_gueltig(route):
    """Alle Abbiegungen der Route (Liste von Punkten) erlaubt?"""
    for i in range(len(route) - 2):
        if not erlaubt(route[i], route[i + 1], route[i + 2]):
            return False
    return True

a, b = (0, 0), (10, 0)
for c in [(20, 0), (10, 10), (0, 5), (3, 1)]:
    print(c, erlaubt(a, b, c))
print(ist_gueltig([(0, 0), (10, 0), (10, 10), (0, 10)]), ist_gueltig([(0, 0), (10, 0), (0, 5)]))
```

Geradeaus (Skalarprodukt −100) und der rechte Winkel (Skalarprodukt 0) sind erlaubt, die beiden Kehrtwenden nicht. Das Quadrat-Stück ist erlaubt (zwei rechte Winkel), das Dreieck `(0, 0), (10, 0), (0, 5)` nicht.

</details>

**Startrouten.** Für die Anfangsroute gibt es kein Verfahren, das immer gelingt (bei einem spitzwinkligen Dreieck gibt es gar keine Route). `route_ab` baut sie mit **Backtracking** ([Notebook 06]): Von einem Startort aus wird immer der nächste noch nicht besuchte Ort angehängt, dessen Abbiegung erlaubt ist; geht es nicht weiter, nimmt sie den letzten Ort zurück und probiert den zweitnächsten. Nach `grenze` Schritten gibt sie auf. Die Funktion ist fertig (gelesen, nicht geschrieben). Die Route ist eine Liste von **Punkten**.

In [ ]:
def route_ab(orte, start, grenze):
    """Backtracking ab dem Ort start (Nummer). Gibt (Route als Liste von Punkten, Schritte) zurück; Route ist None,
    wenn keine gefunden wurde oder die Schrittgrenze erreicht ist."""
    n = len(orte)
    nachbarn = []
    for i in range(n):
        nachbarn.append(sorted(range(n), key=lambda j: math.dist(orte[i], orte[j])))      # nach Abstand, der Ort selbst zuerst
    weg = [start]
    besucht = [False] * n
    besucht[start] = True
    schritte = [0]

    def weiter():
        if len(weg) == n:
            return True
        schritte[0] = schritte[0] + 1
        if schritte[0] > grenze:
            return False
        for j in nachbarn[weg[-1]]:
            if besucht[j]:
                continue
            if len(weg) >= 2 and not erlaubt(orte[weg[-2]], orte[weg[-1]], orte[j]):
                continue
            weg.append(j)
            besucht[j] = True
            if weiter():
                return True
            weg.pop()
            besucht[j] = False
            if schritte[0] > grenze:
                return False
        return False

    if weiter():
        return [orte[i] for i in weg], schritte[0]
    return None, schritte[0]


def weg_laenge(route):
    summe = 0
    for k in range(len(route) - 1):
        summe = summe + math.dist(route[k], route[k + 1])
    return summe

#### Aufgabe E3.2

Schreibe `umkehren(route)`: **2-opt für einen Weg** (kein Kreis, also hat die Route einen Anfang und ein Ende). Für jedes Paar `i < j` von Positionen: Kehrt man das Stück `route[i..j]` um, ändern sich höchstens **zwei Verbindungen**: die vor Position `i` (wenn `i > 0`) und die nach Position `j` (wenn `j < n − 1`). Ist die neue Summe kleiner, baue die neue Route `route[:i] + route[i:j + 1][::-1] + route[j + 1:]` und übernimm sie **nur, wenn `ist_gueltig` gilt**. Wiederhole, bis sich nichts mehr ändert. Teste an der Route, die `route_ab` mit dem Startort 1 findet.

**Erwartete Ausgabe:**
```
1332.0 1205.1 True True
```

In [ ]:
def umkehren(route):
    route = route[:]
    # TODO
    return route

route, schritte = route_ab(orte4, 1, 5000)
besser = umkehren(route)
print(round(weg_laenge(route), 1), round(weg_laenge(besser), 1), ist_gueltig(besser), len(besser) == len(orte4))

<details>
<summary>Tipp anzeigen</summary>

Wie 2-opt in Abschnitt 3, aber ohne den Kreis (kein `% n`) und mit einer zusätzlichen Prüfung `ist_gueltig(neue_route)` vor dem Übernehmen. Die Längenänderung ist immer noch **neu minus alt** aus höchstens zwei Verbindungen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def umkehren(route):
    """2-opt für den Weg: Kehrt Stücke um, wenn der Weg dadurch kürzer wird UND gültig bleibt. Wiederholt, bis nichts mehr geht."""
    route = route[:]
    n = len(route)
    geaendert = True
    while geaendert:
        geaendert = False
        for i in range(n - 1):
            for j in range(i + 1, n):
                alt = 0.0                           # die Verbindung vor Position i und die nach Position j, so wie sie jetzt sind
                neu = 0.0                           # dieselben Verbindungen nach dem Umkehren
                if i > 0:
                    alt = alt + math.dist(route[i - 1], route[i])
                    neu = neu + math.dist(route[i - 1], route[j])
                if j < n - 1:
                    alt = alt + math.dist(route[j], route[j + 1])
                    neu = neu + math.dist(route[i], route[j + 1])
                if neu < alt - 0.000000001:
                    neue_route = route[:i] + route[i:j + 1][::-1] + route[j + 1:]
                    if ist_gueltig(neue_route):
                        route = neue_route
                        geaendert = True
    return route

route, schritte = route_ab(orte4, 1, 5000)
besser = umkehren(route)
print(round(weg_laenge(route), 1), round(weg_laenge(besser), 1), ist_gueltig(besser), len(besser) == len(orte4))
```

Die Route von Startort 1 ist zuerst deutlich länger (1332,0), nach dem Umkehren 1205,1 und weiter gültig. Das Umkehren **innerhalb** des Stücks ändert keine Abbiegung (das Skalarprodukt in `erlaubt` ist in `a` und `c` symmetrisch), nur an den beiden Enden des Stücks kann eine Abbiegung verboten sein. Die Prüfung der ganzen Route ist also mehr als nötig, aber einfach.

</details>

#### Aufgabe E3.3

Starte `route_ab` von **jedem** der 25 Orte aus (mit höchstens 5 000 Schritten je Start), verbessere jede gefundene Route mit `umkehren` und gib aus: von wie vielen Startorten es eine Route gab, die Zahl der Schritte insgesamt, die kürzeste Route vor und nach dem Umkehren. Die Zelle ist fertig, führe sie aus.

**Erwartete Ausgabe:**
```
Startorte mit gefundener Route: 17 von 25  Schritte insgesamt: 49113
kürzeste Route vor dem Umkehren:  1205.07
kürzeste Route nach dem Umkehren: 1205.07
```

In [ ]:
gefunden = 0
schritte_gesamt = 0
vorher = []
nachher = []
for start in range(len(orte4)):
    route, schritte = route_ab(orte4, start, 5000)
    schritte_gesamt = schritte_gesamt + schritte
    if route is not None:
        gefunden = gefunden + 1
        besser = umkehren(route)
        vorher.append(weg_laenge(route))
        nachher.append(weg_laenge(besser))
        assert ist_gueltig(besser) and len(besser) == len(orte4)
print('Startorte mit gefundener Route:', gefunden, 'von', len(orte4), ' Schritte insgesamt:', schritte_gesamt)
print('kürzeste Route vor dem Umkehren: ', round(min(vorher), 2))
print('kürzeste Route nach dem Umkehren:', round(min(nachher), 2))

<details>
<summary>Tipp anzeigen</summary>

Es genügt, die Zelle auszuführen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
gefunden = 0
schritte_gesamt = 0
vorher = []
nachher = []
for start in range(len(orte4)):
    route, schritte = route_ab(orte4, start, 5000)
    schritte_gesamt = schritte_gesamt + schritte
    if route is not None:
        gefunden = gefunden + 1
        besser = umkehren(route)
        vorher.append(weg_laenge(route))
        nachher.append(weg_laenge(besser))
        assert ist_gueltig(besser) and len(besser) == len(orte4)
print('Startorte mit gefundener Route:', gefunden, 'von', len(orte4), ' Schritte insgesamt:', schritte_gesamt)
print('kürzeste Route vor dem Umkehren: ', round(min(vorher), 2))
print('kürzeste Route nach dem Umkehren:', round(min(nachher), 2))
```

</details>

#### Frage E3.4

1. Die Lösungshinweise nennen für dieses Beispiel (`wenigerkrumm4`) die kürzeste Route mit 1205,07 als Optimum (das Programm der Hinweise fand 1205,1). Was sagt dir das über das Ergebnis oben, und was **nicht**?
2. Warum wäre die Anfangsroute **nächster Nachbar ohne Zurücknehmen** (wie in Abschnitt 2) hier keine gute Idee?
3. Die Aufgabe fragt: „Kann das Programm immer eine Route finden?“ Was antwortest du, und wie kannst du das **beweisen**?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Unsere kürzeste Route (1205,07) ist **gleich dem Optimum** der Hinweise: besser geht es nicht. Das gilt aber nur für **dieses** Beispiel mit 25 Orten. Für größere Beispiele ist es nicht gesichert (das Lösungsnotebook bleibt bei den größten Beispielen 9 bis 10 Prozent über dem Optimum der Hinweise).
2. Der nächste Nachbar schaut nur auf die Länge, nicht auf den Winkel. Oft bleibt er **ohne erlaubte Fortsetzung** stecken (der nächste Ort liegt „hinter“ ihm, eine Kehrtwende). Dann muss man **zurücknehmen**, und das ist **Backtracking** wie in Notebook 06.
3. **Nein**, nicht immer. Bei einem **spitzwinkligen Dreieck** (alle Winkel unter 90°) gibt es keine Route: Ist man in einer Ecke und fliegt zu einer anderen, muss man dort mit mehr als 90° abbiegen, um zur dritten zu kommen. Mit vollständigem Backtracking (so viele Schritte, wie nötig) beweist das Programm sogar, dass es keine gibt: Es hat alle Möglichkeiten ausgeschlossen. (Das Lösungsnotebook zeigt das am Dreieck; ein Solver kann es auch mit MiniZinc.)

</details>

## Das hast du gelernt

```python
# 2-opt: Längenänderung in O(1) (nur zwei Kanten ändern sich)
def delta_zwei_opt(d, tour, i, j):
    a, b = tour[i], tour[i + 1]
    c, e = tour[j], tour[(j + 1) % len(tour)]
    return d[a][c] + d[b][e] - d[a][b] - d[c][e]          # negativ = die Tour wird kürzer
# Held-Karp: Mengen als Bitmasken, bester[maske][j] = kürzester Weg über die Orte der Maske nach j
# Eulerkreis (Hierholzer): jede Straße genau einmal, wenn alle Grade gerade sind und der Plan zusammenhängt
```

- Eine **Rundreise** ist eine Liste von Orten, ihre **Länge** und ein **Schiedsrichter** (jeder Ort genau einmal) gehören immer dazu. Durchprobieren geht nur bis `n ≈ 10` (`(n − 1)! / 2` Touren).
- **Aufbau:** nächster Nachbar (O(n²)) und billigste Einfügung (O(n³)) sind schnell, aber oft 5 bis 50 Prozent zu lang. **Verbessern:** 2-opt (Stück umkehren, O(1) je Zug) und Or-opt (Stück verschieben) holen den Großteil des Abstands zurück. Das ist die lokale Suche aus Notebook 04.
- **Exakt für kleine Fälle:** Held-Karp mit **Bitmasken** (`1 << i`, `|`, `&`, `^`) in O(2ⁿ · n²) Zeit und 2ⁿ · n Speicher, praktisch bis `n ≈ 15` in Python. Das TSP ist **NP-schwer**: Für große Fälle gibt es keinen bekannten schnellen exakten Algorithmus, und man braucht Heuristiken.
- **Güte messen:** untere **Schranke** (hier: zwei kürzeste Kanten je Ort, durch 2), das **Optimum kleiner Fälle** (Held-Karp, MiniZinc `circuit`) und eine **Tabelle über viele Startwerte** mit Mittel und schlechtestem Fall. Ohne Optimum nur **relativ** zur besten gefundenen Tour.
- **Über alle Straßen:** Der **Eulerkreis** existiert genau dann, wenn alle Grade gerade sind **und** der Plan zusammenhängt. Sonst **paart** man die Kreuzungen mit ungeradem Grad über **kürzeste Wege** (Dijkstra) und fährt diese Wege **doppelt**. Die Länge ist Summe plus Zusatz. Das ist **nicht** NP-schwer (Blossom-Algorithmus), mit zusätzlichen Bedingungen (Tage, Kapazität, Einbahnstraßen) aber schon.
- Funktionen als Argumente (`STARTS`, `VERBESSERUNGEN`) halten Experimente übersichtlich, und der Schiedsrichter fängt Fehler wie ein falsches Umkehren oder einen unvollständigen Eulerkreis.

### Aufgaben der 2. Runde dazu

Bearbeite eine Aufgabe zuerst selbst und sieh dir das Lösungsnotebook erst danach an.

- [Gießroboter](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/44_Giessroboter/giessroboter_loesung.ipynb) (44. BwInf): Routen durch Einfügen und 2-opt, Aufteilen auf Roboter
- [Müllabfuhr](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/40_Muellabfuhr/muellabfuhr_loesung.ipynb) (40.): Eulerkreis, ungerade Kreuzungen paaren, Aufteilen auf fünf Tage, MiniZinc
- [Weniger krumme Touren](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/41_WenigerKrummeTouren/touren_loesung.ipynb) (41.): Weg mit Winkelbedingung, Backtracking und Umkehren, MiniZinc
- [Ned](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/30_Ned/ned_loesung.ipynb) (30.): jede Straße streuen mit begrenzter Ladung (Briefträgerproblem mit Kapazität)

Weitere Aufgaben mit Touren stehen in der Werkzeugübersicht der 2. Runde (Abschnitt „Heuristiken“, Absatz „Rundreisen und Touren“). Als Nächstes kommt Notebook 08 „Wege mit Nebenbedingungen“: kürzeste Wege, bei denen der Zustand mehr als der Ort ist (Dijkstra auf erweiterten Zuständen).